In [20]:
# batch_attack_counts_with_percentages.py
# 200 randomized 10-task tasksets.
# For each taskset: pick random (attacker, victim) (any order),
# run Vanilla RM and ε-RM, count Anterior/Posterior/Pincer,
# and print TOTAL counts + percentages:
#   %attack = (detected attack instances) / (total attacker execution segments) * 100

from __future__ import annotations
from dataclasses import dataclass
import random
import time
from typing import List, Dict, Tuple, Optional

# =========================
# Experiment knobs
# =========================
N_TRIALS = 200
N_TASKS = 10

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200
U_TOTAL_FIXED = 0.1

EPSILON = 1000

# epsilon inter-arrival acceptance bounds (global)
T_MIN_NS = 10 * 1_000_000   # 10 ms
T_MAX_NS = 200 * 1_000_000  # 200 ms

# Horizon = HORIZON_MULT * Tmax (Tmax is max base period in the taskset)
HORIZON_MULT = 200

# Optional: ensure each generated taskset is RM-schedulable (D=T) via RTA
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 5000

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000

# =========================
# Laplace lookup table (units = 100us), N=100
# Symmetric sampling: u in [0,2N-1], first half positive, second half negative.
# =========================
FLATTENED_LAPLACE_100US: Dict[int, List[int]] = {
    1: [
        121, 726, 1337, 1954, 2578, 3207, 3844, 4487, 5137, 5794,
        6459, 7130, 7809, 8496, 9191, 9894, 10605, 11324, 12052, 12789,
        13534, 14289, 15054, 15828, 16613, 17407, 18212, 19028, 19855, 20693,
        21543, 22405, 23280, 24167, 25068, 25982, 26910, 27852, 28809, 29781,
        30770, 31774, 32796, 33835, 34892, 35967, 37062, 38177, 39313, 40471,
        41651, 42854, 44082, 45335, 46615, 47921, 49257, 50623, 52020, 53449,
        54914, 56414, 57953, 59531, 61151, 62816, 64528, 66289, 68102, 69972,
        71901, 73893, 75952, 78084, 80293, 82586, 84968, 87448, 90032, 92732,
        95557, 98520, 101635, 104918, 108388, 112069, 115986, 120174, 124672, 129529,
        134808, 140589, 146979, 154119, 162211, 171548, 182584, 196077, 213444, 237850
    ],
    10: [
        12, 72, 133, 195, 257, 320, 384, 448, 513, 579,
        645, 713, 780, 849, 919, 989, 1060, 1132, 1205, 1278,
        1353, 1428, 1505, 1582, 1661, 1740, 1821, 1902, 1985, 2069,
        2154, 2240, 2328, 2416, 2506, 2598, 2691, 2785, 2880, 2978,
        3077, 3177, 3279, 3383, 3489, 3596, 3706, 3817, 3931, 4047,
        4165, 4285, 4408, 4533, 4661, 4792, 4925, 5062, 5202, 5344,
        5491, 5641, 5795, 5953, 6115, 6281, 6452, 6628, 6810, 6997,
        7190, 7389, 7595, 7808, 8029, 8258, 8496, 8744, 9003, 9273,
        9555, 9852, 10163, 10491, 10838, 11206, 11598, 12017, 12467, 12952,
        13480, 14058, 14697, 15411, 16221, 17154, 18258, 19607, 21344, 23785
    ],
    100: [
        1, 7, 13, 19, 25, 32, 38, 44, 51, 57,
        64, 71, 78, 84, 91, 98, 106, 113, 120, 127,
        135, 142, 150, 158, 166, 174, 182, 190, 198, 206,
        215, 224, 232, 241, 250, 259, 269, 278, 288, 297,
        307, 317, 327, 338, 348, 359, 370, 381, 393, 404,
        416, 428, 440, 453, 466, 479, 492, 506, 520, 534,
        549, 564, 579, 595, 611, 628, 645, 662, 681, 699,
        719, 738, 759, 780, 802, 825, 849, 874, 900, 927,
        955, 985, 1016, 1049, 1083, 1120, 1159, 1201, 1246, 1295,
        1348, 1405, 1469, 1541, 1622, 1715, 1825, 1960, 2134, 2378
    ],
    1000: [
        0, 0, 1, 1, 2, 3, 3, 4, 5, 5,
        6, 7, 7, 8, 9, 9, 10, 11, 12, 12,
        13, 14, 15, 15, 16, 17, 18, 19, 19, 20,
        21, 22, 23, 24, 25, 25, 26, 27, 28, 29,
        30, 31, 32, 33, 34, 35, 37, 38, 39, 40,
        41, 42, 44, 45, 46, 47, 49, 50, 52, 53,
        54, 56, 57, 59, 61, 62, 64, 66, 68, 69,
        71, 73, 75, 78, 80, 82, 84, 87, 90, 92,
        95, 98, 101, 104, 108, 112, 115, 120, 124, 129,
        134, 140, 146, 154, 162, 171, 182, 196, 213, 237
    ],
}
NS_PER_100US = 100_000  # 100us in ns


def laplace_noise_ns(epsilon: int, rng: random.Random) -> int:
    eps_key = epsilon if epsilon in FLATTENED_LAPLACE_100US else 1000
    arr = FLATTENED_LAPLACE_100US[eps_key]
    n = len(arr)
    u = rng.randrange(0, 2 * n)
    if u < n:
        return arr[u] * NS_PER_100US
    return -arr[u - n] * NS_PER_100US


def randomized_inter_arrival_ns(
    base_period_ns: int,
    epsilon: int,
    rng: random.Random,
    tmin_ns: int,
    tmax_ns: int,
    max_tries: int = 50_000,
) -> int:
    """Inter-arrival = T + noise, truncated to [tmin, tmax] via rejection sampling."""
    tries = 0
    while True:
        tries += 1
        cand = base_period_ns + laplace_noise_ns(epsilon, rng)
        if tmin_ns <= cand <= tmax_ns:
            return cand
        if tries >= max_tries:
            return min(max(base_period_ns, tmin_ns), tmax_ns)


# =========================
# Task / Job / Segment
# =========================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None
    epsilon: int = EPSILON
    offset_ns: int = 0
    T_min_ns: int = T_MIN_NS
    T_max_ns: int = T_MAX_NS

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns


@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int


@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int


def rm_pick(ready: List[Job]) -> Job:
    # RM priority = smaller period first
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))


# =========================
# RM response-time analysis (D=T) to filter schedulable tasksets
# =========================
def rm_rta_schedulable(tasks_sorted_by_T: List[Task]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True


# =========================
# Taskset generator (UUniFast)
# =========================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils


def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))
        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon=EPSILON,
            T_min_ns=T_MIN_NS,
            T_max_ns=T_MAX_NS,
        ))
    return tasks  # already sorted by period


def generate_taskset_maybe_filtered(base_seed: int, trial_idx: int) -> List[Task]:
    rng = random.Random(base_seed + 10_000 * trial_idx)
    tasks = None
    for _ in range(MAX_GEN_ATTEMPTS if FILTER_SCHEDULABLE else 1):
        tasks = generate_taskset_10_rm(rng, U_TOTAL_FIXED)
        if not FILTER_SCHEDULABLE:
            return tasks
        if rm_rta_schedulable(tasks):
            return tasks
    # fallback
    return tasks


# =========================
# Pattern counting + attacker denominator
# =========================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0   # denominator ("total attacker instances")
    attacker_jobs: int = 0       # optional extra denominator


def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    """
    Returns:
      totals (counts + attacker_segments + attacker_jobs),
      ia_total_samples, ia_diff_from_T_samples (epsilon only)
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()

    # segment tracking (merged)
    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                # attacker job counter
                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = randomized_inter_arrival_ns(
                        base_period_ns=t.T_ns,
                        epsilon=t.epsilon,
                        rng=rng,
                        tmin_ns=t.T_min_ns,
                        tmax_ns=t.T_max_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        """
        Merge contiguous same (task, job). If it becomes a NEW segment, update counts.
        """
        nonlocal prev2, prev1, last

        # merge continuation
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # new segment: shift
        prev2, prev1 = prev1, last
        last = seg

        # attacker segment denominator
        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # adjacency counts (need prev1 contiguous)
        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # pincer (strict): prev2 attacker(jobX) -> prev1 victim -> seg attacker(jobX), all contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            # idle gap breaks adjacency chains
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            seg = Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            )
            push_segment(seg)

            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff


def pick_random_pair(tasks: List[Task], rng: random.Random) -> Tuple[str, str]:
    names = [t.name for t in tasks]
    return tuple(rng.sample(names, 2))  # attacker, victim


def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0


def main():
    # Use time-based seed so each script run is different.
    # If you want reproducible experiments, replace this with a fixed integer (e.g., 2026).
    base_seed = time.time_ns()
    print(f"base_seed = {base_seed}")

    pair_rng = random.Random(base_seed + 999)

    totals_vs_all = Totals()
    totals_eps_all = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    for trial in range(N_TRIALS):
        tasks = generate_taskset_maybe_filtered(base_seed, trial)
        attacker, victim = pick_random_pair(tasks, pair_rng)

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=base_seed + 100_000 + trial,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=False,
        )
        totals_vs_all.anterior += vs_tot.anterior
        totals_vs_all.posterior += vs_tot.posterior
        totals_vs_all.pincer += vs_tot.pincer
        totals_vs_all.attacker_segments += vs_tot.attacker_segments
        totals_vs_all.attacker_jobs += vs_tot.attacker_jobs

        # ε-RM
        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=base_seed + 200_000 + trial,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=True,
        )
        totals_eps_all.anterior += eps_tot.anterior
        totals_eps_all.posterior += eps_tot.posterior
        totals_eps_all.pincer += eps_tot.pincer
        totals_eps_all.attacker_segments += eps_tot.attacker_segments
        totals_eps_all.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

    print("\n==================== Summary ====================")
    print(f"Trials: {N_TRIALS}, Tasks per set: {N_TASKS}, U_total={U_TOTAL_FIXED}")
    print(f"Horizon per trial: {HORIZON_MULT} * Tmax")
    print(f"Schedulability filter (RM RTA): {FILTER_SCHEDULABLE}")

    # Vanilla
    print("\n=== Vanilla RM (TOTAL over all trials) ===")
    print(f"Total attacker segments (denominator): {totals_vs_all.attacker_segments}")
    print(f"Total attacker jobs (extra info):      {totals_vs_all.attacker_jobs}")
    print(f"Anterior count:  {totals_vs_all.anterior}   ({pct(totals_vs_all.anterior, totals_vs_all.attacker_segments):.3f}% of attacker segments)")
    print(f"Posterior count: {totals_vs_all.posterior}  ({pct(totals_vs_all.posterior, totals_vs_all.attacker_segments):.3f}% of attacker segments)")
    print(f"Pincer count:    {totals_vs_all.pincer}     ({pct(totals_vs_all.pincer, totals_vs_all.attacker_segments):.3f}% of attacker segments)")

    # Epsilon
    print("\n=== Epsilon RM (TOTAL over all trials) ===")
    print(f"Total attacker segments (denominator): {totals_eps_all.attacker_segments}")
    print(f"Total attacker jobs (extra info):      {totals_eps_all.attacker_jobs}")
    print(f"Anterior count:  {totals_eps_all.anterior}   ({pct(totals_eps_all.anterior, totals_eps_all.attacker_segments):.3f}% of attacker segments)")
    print(f"Posterior count: {totals_eps_all.posterior}  ({pct(totals_eps_all.posterior, totals_eps_all.attacker_segments):.3f}% of attacker segments)")
    print(f"Pincer count:    {totals_eps_all.pincer}     ({pct(totals_eps_all.pincer, totals_eps_all.attacker_segments):.3f}% of attacker segments)")

    # ε inter-arrival variation sanity
    if eps_ia_total_all > 0:
        print("\n=== ε inter-arrival sanity (job-level) ===")
        print(f"Total inter-arrival samples (all tasks, all ε trials): {eps_ia_total_all}")
        print(f"Samples where inter-arrival != base T:                {eps_ia_diff_all}")
        print(f"Fraction varying from T:                              {eps_ia_diff_all/eps_ia_total_all:.4f}")

    print("\nDone.")


if __name__ == "__main__":
    main()

base_seed = 1772231695743546400

==================== Summary ====================
Trials: 200, Tasks per set: 10, U_total=0.1
Horizon per trial: 200 * Tmax
Schedulability filter (RM RTA): True

=== Vanilla RM (TOTAL over all trials) ===
Total attacker segments (denominator): 114316
Total attacker jobs (extra info):      110971
Anterior count:  3102   (2.714% of attacker segments)
Posterior count: 2933  (2.566% of attacker segments)
Pincer count:    393     (0.344% of attacker segments)

=== Epsilon RM (TOTAL over all trials) ===
Total attacker segments (denominator): 112393
Total attacker jobs (extra info):      105887
Anterior count:  1780   (1.584% of attacker segments)
Posterior count: 1758  (1.564% of attacker segments)
Pincer count:    721     (0.641% of attacker segments)

=== ε inter-arrival sanity (job-level) ===
Total inter-arrival samples (all tasks, all ε trials): 1051283
Samples where inter-arrival != base T:                1028764
Fraction varying from T:                 

In [26]:
# epsilon_rm_with_J_delta_eta_batch_attacks.py
# Implements ε-scheduler using bounded Laplace with scale b = (2 * J_i * Δη_i) / ε_i.
# Runs 200 random 10-task tasksets, random attacker/victim each trial, compares Vanilla RM vs ε-RM.
# Prints total Anterior/Posterior/Pincer counts and percentages (per attacker execution segment).

from __future__ import annotations
from dataclasses import dataclass
import math
import random
import time
from typing import List, Dict, Tuple, Optional

# =========================
# Experiment knobs
# =========================
N_TRIALS = 200
N_TASKS = 10

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200
U_TOTAL_FIXED = 0.75

# Horizon = 200 * Tmax (Tmax = max base period in the taskset)
HORIZON_MULT = 200

# Filter tasksets to be schedulable under Vanilla RM (D=T) using RTA
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 5000

# =========================
# ε-scheduler parameters (paper-style)
# =========================
# Bounds for bounded Laplace (T_i^⊥, T_i^⊤)
GLOBAL_T_PERP_MS = 10
GLOBAL_T_TOP_MS  = 200

# Effective protection duration J_i
# (Set this to what your paper uses; e.g., 16 in the kernel table example, or 200 in long runs.)
GLOBAL_J = 16

# Privacy budget ε_i
# You can keep constant, or randomize per task.
GLOBAL_EPSILON = 10.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# Sensitivity Δη_i
# If you want Δη_i = (T_i^⊤ - T_i^⊥), set this True (common choice; e.g., 200-10=190ms).
DELTA_ETA_FROM_BOUNDS = True
# Otherwise, set a fixed sensitivity (in ms):
FIXED_DELTA_ETA_MS = 190.0

# Debug sanity: track how often ε inter-arrival != base T
PRINT_EPS_INTERARRIVAL_SANITY = True

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000

T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS  = GLOBAL_T_TOP_MS  * NS_PER_MS

# =========================
# RM response-time analysis (D=T) for filtering
# =========================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# =========================
# UUniFast task utilization generator
# =========================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0, 1).")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# =========================
# Laplace sampler (continuous) + bounded resampling
# =========================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """
    Sample from Laplace(0, b) using inverse CDF.
    b > 0
    """
    # u in (-0.5, 0.5)
    u = rng.random() - 0.5
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Implements:  \tilde{L}(mu, b, [t_perp, t_top]) via rejection sampling
    where b = (2 * J_i * Δη_i) / ε_i
    """
    if epsilon_i <= 0:
        # fallback: no privacy => no noise
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)  # scale in ns
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    # fallback: clip mu into bounds if acceptance is too unlikely
    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# =========================
# Task / Job / Segment
# =========================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    # ε-scheduler params
    epsilon_i: float = GLOBAL_EPSILON
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    # bounds
    T_perp_ns: int = T_PERP_NS
    T_top_ns: int  = T_TOP_NS

    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# =========================
# Taskset generation
# =========================
def generate_taskset_10_rm(rng: random.Random) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_TOTAL_FIXED, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        if RANDOMIZE_EPSILON_PER_TASK:
            eps_i = float(rng.choice(EPSILON_CHOICES))
        else:
            eps_i = float(GLOBAL_EPSILON)

        if DELTA_ETA_FROM_BOUNDS:
            delta_eta_ns = int(T_TOP_NS - T_PERP_NS)
        else:
            delta_eta_ns = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks  # already sorted by T

def generate_taskset_maybe_filtered(base_seed: int, trial_idx: int) -> List[Task]:
    rng = random.Random(base_seed + 10_000 * trial_idx)
    tasks = None
    for _ in range(MAX_GEN_ATTEMPTS if FILTER_SCHEDULABLE else 1):
        tasks = generate_taskset_10_rm(rng)
        if not FILTER_SCHEDULABLE or rm_rta_schedulable(tasks):
            return tasks
    return tasks

# =========================
# Pattern counting + denominator
# =========================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

def pick_random_pair(tasks: List[Task], rng: random.Random) -> Tuple[str, str]:
    names = [t.name for t in tasks]
    a, v = rng.sample(names, 2)
    return a, v

def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    """
    Returns:
      totals (counts + attacker_segments + attacker_jobs),
      ia_total_samples, ia_diff_from_T_samples (epsilon only)
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()
    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    # Paper-style bounded Laplace inter-arrival
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,                     # η_i(j) = base period
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # shift history
        prev2, prev1 = prev1, last
        last = seg

        # denominator: attacker execution segments
        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # anterior/posterior on contiguous boundary
        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # strict pincer: A(jobX) -> V -> A(jobX), all contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            seg = Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            )
            push_segment(seg)

            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff

# =========================
# Main batch experiment
# =========================
def main():
    # Randomize each script run; print seed so you can reproduce later.
    base_seed = time.time_ns()
    print(f"base_seed = {base_seed}")

    pair_rng = random.Random(base_seed + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    for trial in range(N_TRIALS):
        tasks = generate_taskset_maybe_filtered(base_seed, trial)
        attacker, victim = pick_random_pair(tasks, pair_rng)

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=base_seed + 100_000 + trial,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=False,
        )
        totals_vs.anterior += vs_tot.anterior
        totals_vs.posterior += vs_tot.posterior
        totals_vs.pincer += vs_tot.pincer
        totals_vs.attacker_segments += vs_tot.attacker_segments
        totals_vs.attacker_jobs += vs_tot.attacker_jobs

        # ε-RM (paper-style b = 2 J Δη / ε)
        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=base_seed + 200_000 + trial,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=True,
        )
        totals_eps.anterior += eps_tot.anterior
        totals_eps.posterior += eps_tot.posterior
        totals_eps.pincer += eps_tot.pincer
        totals_eps.attacker_segments += eps_tot.attacker_segments
        totals_eps.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

    print("\n==================== Summary ====================")
    print(f"Trials: {N_TRIALS}, Tasks per set: {N_TASKS}, U_total={U_TOTAL_FIXED}")
    print(f"Horizon per trial: {HORIZON_MULT} * Tmax")
    print(f"Schedulability filter (RM RTA): {FILTER_SCHEDULABLE}")

    print("\n=== Vanilla RM (TOTAL over all trials) ===")
    print(f"Total attacker segments (denominator): {totals_vs.attacker_segments}")
    print(f"Total attacker jobs (extra info):      {totals_vs.attacker_jobs}")
    print(f"Anterior:  {totals_vs.anterior}  ({pct(totals_vs.anterior, totals_vs.attacker_segments):.3f}% of attacker segments)")
    print(f"Posterior: {totals_vs.posterior} ({pct(totals_vs.posterior, totals_vs.attacker_segments):.3f}% of attacker segments)")
    print(f"Pincer:    {totals_vs.pincer}    ({pct(totals_vs.pincer, totals_vs.attacker_segments):.3f}% of attacker segments)")

    print("\n=== Epsilon RM (TOTAL over all trials) ===")
    print(f"ε params used: epsilon={GLOBAL_EPSILON} (randomize={RANDOMIZE_EPSILON_PER_TASK}), J={GLOBAL_J}, "
          f"Δη={'(T_top-T_perp)' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
          f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms")
    print(f"Total attacker segments (denominator): {totals_eps.attacker_segments}")
    print(f"Total attacker jobs (extra info):      {totals_eps.attacker_jobs}")
    print(f"Anterior:  {totals_eps.anterior}  ({pct(totals_eps.anterior, totals_eps.attacker_segments):.3f}% of attacker segments)")
    print(f"Posterior: {totals_eps.posterior} ({pct(totals_eps.posterior, totals_eps.attacker_segments):.3f}% of attacker segments)")
    print(f"Pincer:    {totals_eps.pincer}    ({pct(totals_eps.pincer, totals_eps.attacker_segments):.3f}% of attacker segments)")

    if PRINT_EPS_INTERARRIVAL_SANITY and eps_ia_total_all > 0:
        print("\n=== ε inter-arrival sanity (job-level) ===")
        print(f"Total inter-arrival samples (all tasks, all ε trials): {eps_ia_total_all}")
        print(f"Samples where inter-arrival != base T:                {eps_ia_diff_all}")
        print(f"Fraction varying from T:                              {eps_ia_diff_all/eps_ia_total_all:.4f}")

    print("\nDone.")


if __name__ == "__main__":
    main()

base_seed = 1772237633495020700

==================== Summary ====================
Trials: 200, Tasks per set: 10, U_total=0.75
Horizon per trial: 200 * Tmax
Schedulability filter (RM RTA): True

=== Vanilla RM (TOTAL over all trials) ===
Total attacker segments (denominator): 193790
Total attacker jobs (extra info):      135510
Anterior:  18785  (9.693% of attacker segments)
Posterior: 17396 (8.977% of attacker segments)
Pincer:    5621    (2.901% of attacker segments)

=== Epsilon RM (TOTAL over all trials) ===
ε params used: epsilon=10.0 (randomize=False), J=16, Δη=(T_top-T_perp), bounds=[10,200]ms
Total attacker segments (denominator): 104178
Total attacker jobs (extra info):      69665
Anterior:  8176  (7.848% of attacker segments)
Posterior: 7986 (7.666% of attacker segments)
Pincer:    2843    (2.729% of attacker segments)

=== ε inter-arrival sanity (job-level) ===
Total inter-arrival samples (all tasks, all ε trials): 694964
Samples where inter-arrival != base T:              

## varying utilization

In [59]:
# util_sweep_attack_counts.py
# Runs the SAME experiment for utilization U in [0.1..0.8].
# For each U:
#   - generate 200 RANDOM 10-task RM tasksets (optionally filtered schedulable by RM-RTA, D=T)
#   - pick random attacker/victim per taskset
#   - run Vanilla RM and ε-RM (paper-style bounded Laplace with b = 2*J*Δη/ε)
#   - accumulate total Anterior/Posterior/Pincer counts + percentages (normalized by attacker segments)
#   - save results to a separate CSV file per utilization + one combined summary CSV

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# =========================
# Sweep settings
# =========================
U_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

N_TRIALS = 200
N_TASKS = 5+5+5+5

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200

# Horizon = 200 * Tmax (Tmax = max base period in each taskset)
HORIZON_MULT = 200

# =========================
# ε-scheduler parameters (paper-style)
# =========================
# GLOBAL_T_PERP_MS = 10
# GLOBAL_T_TOP_MS  = 200
GLOBAL_T_PERP_MS = 10
GLOBAL_T_TOP_MS  = 200
GLOBAL_J = 16

GLOBAL_EPSILON = 1.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# If True, Δη = (T_top - T_perp) (common: 200-10 = 190ms)
DELTA_ETA_FROM_BOUNDS = True
FIXED_DELTA_ETA_MS = 190.0

# =========================
# Taskset filtering
# =========================
FILTER_SCHEDULABLE = True          # filter by RM response-time analysis with D=T
MAX_GEN_ATTEMPTS = 8000            # per trial (if too small at high U, increase)

# =========================
# Output
# =========================
OUT_DIR = Path("util_sweep_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000
T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS  = GLOBAL_T_TOP_MS  * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# ============================================================
# RM response-time analysis (D=T) for filtering tasksets
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# ============================================================
# Laplace (continuous) sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5  # (-0.5, 0.5)
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Paper-style bounded Laplace:
      b = (2 * J_i * Δη_i) / ε_i
      sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# ============================================================
# Task / Job / Segment
# ============================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    # epsilon scheduler params
    epsilon_i: float = GLOBAL_EPSILON
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    # bounds
    T_perp_ns: int = T_PERP_NS
    T_top_ns: int  = T_TOP_NS

    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# Taskset generation (10 tasks, sorted by period)
# ============================================================
def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)
        delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks

def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if filtering enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True
    return last, False  # failed to find schedulable within attempts

# ============================================================
# Pattern counting totals
# ============================================================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

# ============================================================
# Simulation: count anterior/posterior/pincer + attacker denominator
# ============================================================
def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    """
    Returns:
      totals (counts + attacker_segments + attacker_jobs),
      ia_total_samples, ia_diff_from_T_samples (epsilon only)
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()

    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # shift history
        prev2, prev1 = prev1, last
        last = seg

        # denominator: attacker execution segments
        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # anterior/posterior (contiguous)
        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # strict pincer: A(jobX) -> V -> A(jobX), all contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff

# ============================================================
# One utilization run -> returns two summary rows (VS + EPS)
# ============================================================
def run_for_utilization(U_total: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object]]:
    pair_rng = random.Random(global_seed + int(U_total * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed_tasksets = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        # if generation fails (schedulability filter), we bump seeds and retry same "used index"
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, U_total)
        if not ok:
            failed_tasksets += 1
            extra_seed_bump += 1_000_000
            continue

        attacker, victim = random.sample([t.name for t in tasks], 2)
        # t1, t2 = pair_rng.sample(tasks, 2)

        # if t1.T_ns < t2.T_ns:
        #     attacker = t1.name
        #     victim = t2.name
        # else:
        #     attacker = t2.name
        #     victim = t1.name

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=False,
        )
        totals_vs.anterior += vs_tot.anterior
        totals_vs.posterior += vs_tot.posterior
        totals_vs.pincer += vs_tot.pincer
        totals_vs.attacker_segments += vs_tot.attacker_segments
        totals_vs.attacker_jobs += vs_tot.attacker_jobs

        # ε-RM
        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=True,
        )
        totals_eps.anterior += eps_tot.anterior
        totals_eps.posterior += eps_tot.posterior
        totals_eps.pincer += eps_tot.pincer
        totals_eps.attacker_segments += eps_tot.attacker_segments
        totals_eps.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        used += 1

    # Build row dicts
    common = {
        "utilization": U_total,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets,
        "seed": global_seed,
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "attacker_segments": totals_vs.attacker_segments,
        "attacker_jobs": totals_vs.attacker_jobs,
        "anterior": totals_vs.anterior,
        "posterior": totals_vs.posterior,
        "pincer": totals_vs.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_vs.anterior, totals_vs.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_vs.posterior, totals_vs.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_vs.pincer, totals_vs.attacker_segments),
        "segments_per_job": (totals_vs.attacker_segments / totals_vs.attacker_jobs) if totals_vs.attacker_jobs else 0.0,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={'T_top-T_perp' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "attacker_segments": totals_eps.attacker_segments,
        "attacker_jobs": totals_eps.attacker_jobs,
        "anterior": totals_eps.anterior,
        "posterior": totals_eps.posterior,
        "pincer": totals_eps.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_eps.anterior, totals_eps.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_eps.posterior, totals_eps.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_eps.pincer, totals_eps.attacker_segments),
        "segments_per_job": (totals_eps.attacker_segments / totals_eps.attacker_jobs) if totals_eps.attacker_jobs else 0.0,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps

# ============================================================
# Main
# ============================================================
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    # stable field ordering: union of keys
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                seen.add(k)
                keys.append(k)

    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def main():
    global_seed = time.time_ns()
    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    all_rows: List[Dict[str, object]] = []

    for U in U_LIST:
        print(f"\n--- Running utilization U={U:.1f} ---")
        row_vs, row_eps = run_for_utilization(U, global_seed)

        # console summary
        print("Vanilla RM:",
              f"A={row_vs['anterior']} ({row_vs['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_vs['posterior']} ({row_vs['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_vs['pincer']} ({row_vs['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_vs['attacker_segments']}")
        print("Epsilon RM:",
              f"A={row_eps['anterior']} ({row_eps['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_eps['posterior']} ({row_eps['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_eps['pincer']} ({row_eps['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_eps['attacker_segments']}",
              f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}")

        # save per-utilization
        util_tag = int(round(U * 100))
        per_u_path = OUT_DIR / f"results_U{util_tag:02d}.csv"
        write_csv(per_u_path, [row_vs, row_eps])

        all_rows.extend([row_vs, row_eps])

    # save combined summary
    summary_path = OUT_DIR / "summary_all_utilizations.csv"
    write_csv(summary_path, all_rows)

    print("\nDone.")
    print(f"Combined summary: {summary_path.resolve()}")

if __name__ == "__main__":
    main()

global_seed = 1772498661797516500
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\util_sweep_results

--- Running utilization U=0.1 ---
Vanilla RM: A=3276 (2.399%), P=1495 (1.095%), Pin=127 (0.093%), den=136540
Epsilon RM: A=618 (0.795%), P=594 (0.764%), Pin=241 (0.310%), den=77742 IA-diff-frac=1.0000

--- Running utilization U=0.2 ---
Vanilla RM: A=3411 (2.434%), P=2354 (1.680%), Pin=376 (0.268%), den=140118
Epsilon RM: A=1191 (1.425%), P=1306 (1.562%), Pin=546 (0.653%), den=83606 IA-diff-frac=1.0000

--- Running utilization U=0.3 ---
Vanilla RM: A=3076 (2.126%), P=2257 (1.560%), Pin=477 (0.330%), den=144712
Epsilon RM: A=1639 (1.896%), P=1571 (1.818%), Pin=560 (0.648%), den=86429 IA-diff-frac=1.0000

--- Running utilization U=0.4 ---
Vanilla RM: A=3981 (2.548%), P=3998 (2.559%), Pin=651 (0.417%), den=156223
Epsilon RM: A=2433 (2.733%), P=2139 (2.402%), Pin=647 (0.727%), den=89037 IA-diff-frac=1.0000

# vary utilization.. attacker has HP than Victim


In [ ]:
# util_sweep_attack_counts.py
# Runs the SAME experiment for utilization U in [0.1..0.8].
# For each U:
#   - generate 200 RANDOM 10-task RM tasksets (optionally filtered schedulable by RM-RTA, D=T)
#   - pick random attacker/victim per taskset
#   - run Vanilla RM and ε-RM (paper-style bounded Laplace with b = 2*J*Δη/ε)
#   - accumulate total Anterior/Posterior/Pincer counts + percentages (normalized by attacker segments)
#   - save results to a separate CSV file per utilization + one combined summary CSV

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# =========================
# Sweep settings
# =========================
U_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

N_TRIALS = 500
N_TASKS = 10

PERIOD_MIN_MS = 50
PERIOD_MAX_MS = 150

# Horizon = 200 * Tmax (Tmax = max base period in each taskset)
HORIZON_MULT = 200

# =========================
# ε-scheduler parameters (paper-style)
# =========================
# GLOBAL_T_PERP_MS = 10
# GLOBAL_T_TOP_MS  = 200
GLOBAL_T_PERP_MS = 50
GLOBAL_T_TOP_MS  = 150
GLOBAL_J = 10

GLOBAL_EPSILON = 10.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# If True, Δη = (T_top - T_perp) (common: 200-10 = 190ms)
DELTA_ETA_FROM_BOUNDS = True
FIXED_DELTA_ETA_MS = 190.0

# =========================
# Taskset filtering
# =========================
FILTER_SCHEDULABLE = True          # filter by RM response-time analysis with D=T
MAX_GEN_ATTEMPTS = 8000            # per trial (if too small at high U, increase)

# =========================
# Output
# =========================
OUT_DIR = Path("util_sweep_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000
T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS  = GLOBAL_T_TOP_MS  * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# ============================================================
# RM response-time analysis (D=T) for filtering tasksets
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# ============================================================
# Laplace (continuous) sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5  # (-0.5, 0.5)
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Paper-style bounded Laplace:
      b = (2 * J_i * Δη_i) / ε_i
      sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# ============================================================
# Task / Job / Segment
# ============================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    # epsilon scheduler params
    epsilon_i: float = GLOBAL_EPSILON
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    # bounds
    T_perp_ns: int = T_PERP_NS
    T_top_ns: int  = T_TOP_NS

    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# Taskset generation (10 tasks, sorted by period)
# ============================================================
def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)
        delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks

def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if filtering enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True
    return last, False  # failed to find schedulable within attempts

# ============================================================
# Pattern counting totals
# ============================================================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

# ============================================================
# Simulation: count anterior/posterior/pincer + attacker denominator
# ============================================================
def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    """
    Returns:
      totals (counts + attacker_segments + attacker_jobs),
      ia_total_samples, ia_diff_from_T_samples (epsilon only)
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()

    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # shift history
        prev2, prev1 = prev1, last
        last = seg

        # denominator: attacker execution segments
        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # anterior/posterior (contiguous)
        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # strict pincer: A(jobX) -> V -> A(jobX), all contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff

# ============================================================
# One utilization run -> returns two summary rows (VS + EPS)
# ============================================================
def run_for_utilization(U_total: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object]]:
    pair_rng = random.Random(global_seed + int(U_total * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed_tasksets = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        # if generation fails (schedulability filter), we bump seeds and retry same "used index"
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, U_total)
        if not ok:
            failed_tasksets += 1
            extra_seed_bump += 1_000_000
            continue

        attacker, victim = random.sample([t.name for t in tasks], 2)
        # t1, t2 = pair_rng.sample(tasks, 2)

        # if t1.T_ns < t2.T_ns:
        #     attacker = t1.name
        #     victim = t2.name
        # else:
        #     attacker = t2.name
        #     victim = t1.name

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=False,
        )
        totals_vs.anterior += vs_tot.anterior
        totals_vs.posterior += vs_tot.posterior
        totals_vs.pincer += vs_tot.pincer
        totals_vs.attacker_segments += vs_tot.attacker_segments
        totals_vs.attacker_jobs += vs_tot.attacker_jobs

        # ε-RM
        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=True,
        )
        totals_eps.anterior += eps_tot.anterior
        totals_eps.posterior += eps_tot.posterior
        totals_eps.pincer += eps_tot.pincer
        totals_eps.attacker_segments += eps_tot.attacker_segments
        totals_eps.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        used += 1

    # Build row dicts
    common = {
        "utilization": U_total,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets,
        "seed": global_seed,
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "attacker_segments": totals_vs.attacker_segments,
        "attacker_jobs": totals_vs.attacker_jobs,
        "anterior": totals_vs.anterior,
        "posterior": totals_vs.posterior,
        "pincer": totals_vs.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_vs.anterior, totals_vs.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_vs.posterior, totals_vs.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_vs.pincer, totals_vs.attacker_segments),
        "segments_per_job": (totals_vs.attacker_segments / totals_vs.attacker_jobs) if totals_vs.attacker_jobs else 0.0,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={'T_top-T_perp' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "attacker_segments": totals_eps.attacker_segments,
        "attacker_jobs": totals_eps.attacker_jobs,
        "anterior": totals_eps.anterior,
        "posterior": totals_eps.posterior,
        "pincer": totals_eps.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_eps.anterior, totals_eps.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_eps.posterior, totals_eps.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_eps.pincer, totals_eps.attacker_segments),
        "segments_per_job": (totals_eps.attacker_segments / totals_eps.attacker_jobs) if totals_eps.attacker_jobs else 0.0,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps

# ============================================================
# Main
# ============================================================
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    # stable field ordering: union of keys
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                seen.add(k)
                keys.append(k)

    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def main():
    global_seed = time.time_ns()
    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    all_rows: List[Dict[str, object]] = []

    for U in U_LIST:
        print(f"\n--- Running utilization U={U:.1f} ---")
        row_vs, row_eps = run_for_utilization(U, global_seed)

        # console summary
        print("Vanilla RM:",
              f"A={row_vs['anterior']} ({row_vs['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_vs['posterior']} ({row_vs['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_vs['pincer']} ({row_vs['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_vs['attacker_segments']}")
        print("Epsilon RM:",
              f"A={row_eps['anterior']} ({row_eps['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_eps['posterior']} ({row_eps['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_eps['pincer']} ({row_eps['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_eps['attacker_segments']}",
              f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}")

        # save per-utilization
        util_tag = int(round(U * 100))
        per_u_path = OUT_DIR / f"results_U{util_tag:02d}.csv"
        write_csv(per_u_path, [row_vs, row_eps])

        all_rows.extend([row_vs, row_eps])

    # save combined summary
    summary_path = OUT_DIR / "summary_all_utilizations.csv"
    write_csv(summary_path, all_rows)

    print("\nDone.")
    print(f"Combined summary: {summary_path.resolve()}")

if __name__ == "__main__":
    main()

global_seed = 1772306201034171300
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\util_sweep_results

--- Running utilization U=0.1 ---
Vanilla RM: A=5266 (2.815%), P=870 (0.465%), Pin=0 (0.000%), den=187043
Epsilon RM: A=2896 (1.953%), P=1616 (1.090%), Pin=0 (0.000%), den=148250 IA-diff-frac=1.0000

--- Running utilization U=0.2 ---
Vanilla RM: A=8317 (4.298%), P=2458 (1.270%), Pin=0 (0.000%), den=193509
Epsilon RM: A=5787 (3.793%), P=3272 (2.145%), Pin=0 (0.000%), den=152572 IA-diff-frac=1.0000

--- Running utilization U=0.3 ---
Vanilla RM: A=10110 (5.180%), P=4023 (2.061%), Pin=0 (0.000%), den=195159
Epsilon RM: A=8806 (5.589%), P=4919 (3.122%), Pin=0 (0.000%), den=157563 IA-diff-frac=1.0000

--- Running utilization U=0.4 ---
Vanilla RM: A=13478 (6.571%), P=6050 (2.950%), Pin=0 (0.000%), den=205114
Epsilon RM: A=11130 (7.010%), P=6591 (4.151%), Pin=0 (0.000%), den=158779 IA-diff-frac=1.0000

--- Ru

# vary itilization Victim has HP than Attacker

In [42]:
# util_sweep_attack_counts.py
# Runs the SAME experiment for utilization U in [0.1..0.8].
# For each U:
#   - generate 200 RANDOM 10-task RM tasksets (optionally filtered schedulable by RM-RTA, D=T)
#   - pick random attacker/victim per taskset
#   - run Vanilla RM and ε-RM (paper-style bounded Laplace with b = 2*J*Δη/ε)
#   - accumulate total Anterior/Posterior/Pincer counts + percentages (normalized by attacker segments)
#   - save results to a separate CSV file per utilization + one combined summary CSV

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# =========================
# Sweep settings
# =========================
U_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

N_TRIALS = 200
N_TASKS = 10

PERIOD_MIN_MS = 50
PERIOD_MAX_MS = 150

# Horizon = 200 * Tmax (Tmax = max base period in each taskset)
HORIZON_MULT = 500

# =========================
# ε-scheduler parameters (paper-style)
# =========================
GLOBAL_T_PERP_MS = 50
GLOBAL_T_TOP_MS  = 150
GLOBAL_J = 10

GLOBAL_EPSILON = 10
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# If True, Δη = (T_top - T_perp) (common: 200-10 = 190ms)
DELTA_ETA_FROM_BOUNDS = True
FIXED_DELTA_ETA_MS = 190.0

# =========================
# Taskset filtering
# =========================
FILTER_SCHEDULABLE = True          # filter by RM response-time analysis with D=T
MAX_GEN_ATTEMPTS = 8000            # per trial (if too small at high U, increase)

# =========================
# Output
# =========================
OUT_DIR = Path("util_sweep_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000
T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS  = GLOBAL_T_TOP_MS  * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# ============================================================
# RM response-time analysis (D=T) for filtering tasksets
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# ============================================================
# Laplace (continuous) sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5  # (-0.5, 0.5)
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Paper-style bounded Laplace:
      b = (2 * J_i * Δη_i) / ε_i
      sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# ============================================================
# Task / Job / Segment
# ============================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    # epsilon scheduler params
    epsilon_i: float = GLOBAL_EPSILON
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    # bounds
    T_perp_ns: int = T_PERP_NS
    T_top_ns: int  = T_TOP_NS

    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# Taskset generation (10 tasks, sorted by period)
# ============================================================
def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)
        delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks

def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if filtering enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True
    return last, False  # failed to find schedulable within attempts

# ============================================================
# Pattern counting totals
# ============================================================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

# ============================================================
# Simulation: count anterior/posterior/pincer + attacker denominator
# ============================================================
def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    """
    Returns:
      totals (counts + attacker_segments + attacker_jobs),
      ia_total_samples, ia_diff_from_T_samples (epsilon only)
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()

    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # shift history
        prev2, prev1 = prev1, last
        last = seg

        # denominator: attacker execution segments
        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # anterior/posterior (contiguous)
        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # strict pincer: A(jobX) -> V -> A(jobX), all contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff

# ============================================================
# One utilization run -> returns two summary rows (VS + EPS)
# ============================================================
def run_for_utilization(U_total: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object]]:
    pair_rng = random.Random(global_seed + int(U_total * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed_tasksets = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        # if generation fails (schedulability filter), we bump seeds and retry same "used index"
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, U_total)
        if not ok:
            failed_tasksets += 1
            extra_seed_bump += 1_000_000
            continue

        #attacker, victim = random.sample([t.name for t in tasks], 2)
        t1, t2 = pair_rng.sample(tasks, 2)

        if t1.T_ns > t2.T_ns:
            attacker = t1.name   # larger period => lower priority
            victim = t2.name     # smaller period => higher priority
        else:
            attacker = t2.name
            victim = t1.name
            
        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=False,
        )
        totals_vs.anterior += vs_tot.anterior
        totals_vs.posterior += vs_tot.posterior
        totals_vs.pincer += vs_tot.pincer
        totals_vs.attacker_segments += vs_tot.attacker_segments
        totals_vs.attacker_jobs += vs_tot.attacker_jobs

        # ε-RM
        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker,
            victim=victim,
            collect_ia_stats=True,
        )
        totals_eps.anterior += eps_tot.anterior
        totals_eps.posterior += eps_tot.posterior
        totals_eps.pincer += eps_tot.pincer
        totals_eps.attacker_segments += eps_tot.attacker_segments
        totals_eps.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        used += 1

    # Build row dicts
    common = {
        "utilization": U_total,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets,
        "seed": global_seed,
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "attacker_segments": totals_vs.attacker_segments,
        "attacker_jobs": totals_vs.attacker_jobs,
        "anterior": totals_vs.anterior,
        "posterior": totals_vs.posterior,
        "pincer": totals_vs.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_vs.anterior, totals_vs.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_vs.posterior, totals_vs.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_vs.pincer, totals_vs.attacker_segments),
        "segments_per_job": (totals_vs.attacker_segments / totals_vs.attacker_jobs) if totals_vs.attacker_jobs else 0.0,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={'T_top-T_perp' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "attacker_segments": totals_eps.attacker_segments,
        "attacker_jobs": totals_eps.attacker_jobs,
        "anterior": totals_eps.anterior,
        "posterior": totals_eps.posterior,
        "pincer": totals_eps.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_eps.anterior, totals_eps.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_eps.posterior, totals_eps.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_eps.pincer, totals_eps.attacker_segments),
        "segments_per_job": (totals_eps.attacker_segments / totals_eps.attacker_jobs) if totals_eps.attacker_jobs else 0.0,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps

# ============================================================
# Main
# ============================================================
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    # stable field ordering: union of keys
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                seen.add(k)
                keys.append(k)

    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def main():
    global_seed = time.time_ns()
    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    all_rows: List[Dict[str, object]] = []

    for U in U_LIST:
        print(f"\n--- Running utilization U={U:.1f} ---")
        row_vs, row_eps = run_for_utilization(U, global_seed)

        # console summary
        print("Vanilla RM:",
              f"A={row_vs['anterior']} ({row_vs['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_vs['posterior']} ({row_vs['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_vs['pincer']} ({row_vs['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_vs['attacker_segments']}")
        print("Epsilon RM:",
              f"A={row_eps['anterior']} ({row_eps['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_eps['posterior']} ({row_eps['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_eps['pincer']} ({row_eps['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_eps['attacker_segments']}",
              f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}")

        # save per-utilization
        util_tag = int(round(U * 100))
        per_u_path = OUT_DIR / f"results_U{util_tag:02d}.csv"
        write_csv(per_u_path, [row_vs, row_eps])

        all_rows.extend([row_vs, row_eps])

    # save combined summary
    summary_path = OUT_DIR / "summary_all_utilizations.csv"
    write_csv(summary_path, all_rows)

    print("\nDone.")
    print(f"Combined summary: {summary_path.resolve()}")

if __name__ == "__main__":
    main()

global_seed = 1772325778228861100
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\util_sweep_results

--- Running utilization U=0.1 ---
Vanilla RM: A=852 (0.653%), P=6854 (5.251%), Pin=808 (0.619%), den=130521
Epsilon RM: A=1738 (1.146%), P=3007 (1.983%), Pin=1667 (1.099%), den=151626 IA-diff-frac=1.0000

--- Running utilization U=0.2 ---
Vanilla RM: A=2545 (1.779%), P=7525 (5.260%), Pin=2261 (1.580%), den=143069
Epsilon RM: A=3617 (2.201%), P=6161 (3.749%), Pin=3244 (1.974%), den=164333 IA-diff-frac=1.0000

--- Running utilization U=0.3 ---
Vanilla RM: A=3886 (2.542%), P=10607 (6.938%), Pin=3323 (2.174%), den=152878
Epsilon RM: A=4943 (2.859%), P=8459 (4.893%), Pin=4350 (2.516%), den=172887 IA-diff-frac=1.0000

--- Running utilization U=0.4 ---
Vanilla RM: A=5943 (3.622%), P=13396 (8.163%), Pin=5018 (3.058%), den=164102
Epsilon RM: A=7009 (3.808%), P=11371 (6.177%), Pin=5918 (3.215%), den=184083 IA-d

# varying epsilon

In [47]:
# epsilon_sweep_attack_counts.py
# Fix utilization U=0.5 and n_tasks=10; sweep epsilon over:
# [200,400,600,800,1000,1200,1400,1600,1800,2000]
# For each epsilon:
#   - generate 200 randomized 10-task RM tasksets (optionally filtered schedulable under RM-RTA, D=T)
#   - pick random attacker/victim per taskset
#   - run Vanilla RM (same across eps) and ε-RM (paper-style bounded Laplace with b = 2*J*Δη/ε)
#   - accumulate total Anterior/Posterior/Pincer counts + percentages (normalized by attacker segments)
# Saves one CSV per epsilon + one combined summary CSV.

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# =========================
# Fixed experiment settings
# =========================
U_TOTAL = 0.5
N_TRIALS = 200
N_TASKS = 10

EPSILON_LIST = [1,10,100,1000]

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200

HORIZON_MULT = 200  # horizon per trial = 200 * Tmax

# =========================
# ε-scheduler params (paper-style)
# =========================
GLOBAL_T_PERP_MS = 10
GLOBAL_T_TOP_MS  = 200
GLOBAL_J = 16

DELTA_ETA_FROM_BOUNDS = True        # Δη = (T_top - T_perp) = 190ms
FIXED_DELTA_ETA_MS = 190.0          # used only if above is False

# =========================
# Taskset filtering
# =========================
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 8000

# =========================
# Output
# =========================
OUT_DIR = Path("epsilon_sweep_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000
T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS  = GLOBAL_T_TOP_MS  * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# ============================================================
# RM response-time analysis (D=T) for filtering tasksets
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# ============================================================
# Laplace (continuous) sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    u = rng.random() - 0.5
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    b = (2 * J_i * Δη_i) / ε_i
    sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# ============================================================
# Task / Job / Segment
# ============================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    epsilon_i: float = 1000.0
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(190.0 * NS_PER_MS)

    T_perp_ns: int = T_PERP_NS
    T_top_ns: int  = T_TOP_NS
    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# Taskset generation
# ============================================================
def generate_taskset_10_rm(rng: random.Random, epsilon_value: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_TOTAL, rng)

    delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=float(epsilon_value),
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks

def generate_taskset_filtered(base_seed: int, trial_idx: int, epsilon_value: float) -> Tuple[List[Task], bool]:
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, epsilon_value)
        if not FILTER_SCHEDULABLE or rm_rta_schedulable(last):
            return last, True
    return last, False

# ============================================================
# Totals + simulation counting
# ============================================================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

def pick_random_pair(tasks: List[Task], rng: random.Random) -> Tuple[str, str]:
    names = [t.name for t in tasks]
    return tuple(rng.sample(names, 2))

def simulate_count_patterns_and_denominator(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[Totals, int, int]:
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()
    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    ia_total = 0
    ia_diff = 0

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        # shift history
        prev2, prev1 = prev1, last
        last = seg

        if seg.task_name == attacker:
            totals.attacker_segments += 1

        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals, ia_total, ia_diff

# ============================================================
# Run one epsilon value and save results
# ============================================================
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    keys = []
    seen = set()
    for r in rows:
        for k in r.keys():
            if k not in seen:
                seen.add(k)
                keys.append(k)
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def run_for_epsilon(eps: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object]]:
    pair_rng = random.Random(global_seed + int(eps) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, eps)
        if not ok:
            failed += 1
            extra_seed_bump += 1_000_000
            continue

        attacker, victim = pick_random_pair(tasks, pair_rng)

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        vs_tot, _, _ = simulate_count_patterns_and_denominator(
            tasks, horizon_ns, False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker, victim=victim,
            collect_ia_stats=False
        )
        totals_vs.anterior += vs_tot.anterior
        totals_vs.posterior += vs_tot.posterior
        totals_vs.pincer += vs_tot.pincer
        totals_vs.attacker_segments += vs_tot.attacker_segments
        totals_vs.attacker_jobs += vs_tot.attacker_jobs

        eps_tot, ia_total, ia_diff = simulate_count_patterns_and_denominator(
            tasks, horizon_ns, True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker, victim=victim,
            collect_ia_stats=True
        )
        totals_eps.anterior += eps_tot.anterior
        totals_eps.posterior += eps_tot.posterior
        totals_eps.pincer += eps_tot.pincer
        totals_eps.attacker_segments += eps_tot.attacker_segments
        totals_eps.attacker_jobs += eps_tot.attacker_jobs

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        used += 1

    common = {
        "epsilon": eps,
        "utilization": U_TOTAL,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed,
        "J": GLOBAL_J,
        "delta_eta_ms": (GLOBAL_T_TOP_MS - GLOBAL_T_PERP_MS) if DELTA_ETA_FROM_BOUNDS else FIXED_DELTA_ETA_MS,
        "bounds_ms": f"[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]",
        "seed": global_seed,
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "attacker_segments": totals_vs.attacker_segments,
        "attacker_jobs": totals_vs.attacker_jobs,
        "anterior": totals_vs.anterior,
        "posterior": totals_vs.posterior,
        "pincer": totals_vs.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_vs.anterior, totals_vs.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_vs.posterior, totals_vs.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_vs.pincer, totals_vs.attacker_segments),
        "segments_per_job": (totals_vs.attacker_segments / totals_vs.attacker_jobs) if totals_vs.attacker_jobs else 0.0,
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "attacker_segments": totals_eps.attacker_segments,
        "attacker_jobs": totals_eps.attacker_jobs,
        "anterior": totals_eps.anterior,
        "posterior": totals_eps.posterior,
        "pincer": totals_eps.pincer,
        "anterior_pct_of_attacker_segments": pct(totals_eps.anterior, totals_eps.attacker_segments),
        "posterior_pct_of_attacker_segments": pct(totals_eps.posterior, totals_eps.attacker_segments),
        "pincer_pct_of_attacker_segments": pct(totals_eps.pincer, totals_eps.attacker_segments),
        "segments_per_job": (totals_eps.attacker_segments / totals_eps.attacker_jobs) if totals_eps.attacker_jobs else 0.0,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps

# ============================================================
# Main
# ============================================================
def main():
    global_seed = time.time_ns()
    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    all_rows: List[Dict[str, object]] = []

    for eps in EPSILON_LIST:
        print(f"\n--- Running epsilon={eps} ---")
        row_vs, row_eps = run_for_epsilon(float(eps), global_seed)

        # console summary
        print("Vanilla RM:",
              f"A={row_vs['anterior']} ({row_vs['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_vs['posterior']} ({row_vs['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_vs['pincer']} ({row_vs['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_vs['attacker_segments']}")
        print("Epsilon RM:",
              f"A={row_eps['anterior']} ({row_eps['anterior_pct_of_attacker_segments']:.3f}%),",
              f"P={row_eps['posterior']} ({row_eps['posterior_pct_of_attacker_segments']:.3f}%),",
              f"Pin={row_eps['pincer']} ({row_eps['pincer_pct_of_attacker_segments']:.3f}%),",
              f"den={row_eps['attacker_segments']}",
              f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}")

        # save per-epsilon
        per_eps_path = OUT_DIR / f"results_eps{int(eps):04d}.csv"
        write_csv(per_eps_path, [row_vs, row_eps])
        all_rows.extend([row_vs, row_eps])

    # save combined summary
    summary_path = OUT_DIR / "summary_all_epsilons.csv"
    write_csv(summary_path, all_rows)

    print("\nDone.")
    print(f"Combined summary: {summary_path.resolve()}")

if __name__ == "__main__":
    main()

global_seed = 1772353703080210100
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\epsilon_sweep_results

--- Running epsilon=1 ---
Vanilla RM: A=9371 (6.535%), P=8309 (5.794%), Pin=2603 (1.815%), den=143407
Epsilon RM: A=5615 (6.209%), P=5687 (6.288%), Pin=1836 (2.030%), den=90438 IA-diff-frac=1.0000

--- Running epsilon=10 ---
Vanilla RM: A=11090 (7.611%), P=10797 (7.410%), Pin=3463 (2.377%), den=145705
Epsilon RM: A=6404 (6.567%), P=6208 (6.366%), Pin=2356 (2.416%), den=97513 IA-diff-frac=1.0000

--- Running epsilon=100 ---
Vanilla RM: A=9141 (5.921%), P=9852 (6.381%), Pin=2495 (1.616%), den=154389
Epsilon RM: A=5550 (5.792%), P=5459 (5.697%), Pin=1601 (1.671%), den=95815 IA-diff-frac=1.0000

--- Running epsilon=1000 ---
Vanilla RM: A=9703 (6.850%), P=9840 (6.947%), Pin=3340 (2.358%), den=141646
Epsilon RM: A=9377 (6.613%), P=9572 (6.751%), Pin=3756 (2.649%), den=141788 IA-diff-frac=1.0000

Done.
Co

# vary delta eta

In [60]:
# delta_eta_sweep_attack_plot.py
# Fix: U=0.5, n_tasks=10; sweep delta_eta in ms:
#   [20,40,60,80,100,120,140,160,180,200]
# Compare Vanilla RM vs ε-RM (paper-style bounded Laplace with b = 2*J*Δη/ε).
# Output: CSV summary + PDF/PNG plot.

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import matplotlib.pyplot as plt

# =========================
# Fixed settings
# =========================
U_TOTAL = 0.5
N_TASKS = 10
N_TRIALS = 200

DELTA_ETA_LIST_MS = [0,20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

# Period range (ms)
PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200

# Horizon per trial = 200 * Tmax (Tmax is max base period in that taskset)
HORIZON_MULT = 200

# ε-scheduler params
EPSILON = 1.0
J = 16

# Bounded Laplace interval [T_perp, T_top]
T_PERP_MS = 10
T_TOP_MS  = 200

# Filter tasksets that are schedulable under Vanilla RM (D=T)
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 8000

# Output
OUT_DIR = Path("delta_eta_sweep_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000

T_PERP_NS = T_PERP_MS * NS_PER_MS
T_TOP_NS  = T_TOP_MS  * NS_PER_MS

# =========================
# Helpers
# =========================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# =========================
# RM response-time analysis (D=T)
# =========================
def rm_rta_schedulable(tasks_sorted_by_T: List["TaskBase"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.T_ns  # D=T
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]
        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference
            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next
        if R > D:
            return False
    return True

# =========================
# UUniFast utilization generation
# =========================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# =========================
# Laplace sampler + bounded resampling
# =========================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Paper-style bounded Laplace:
      b = (2 * J_i * Δη_i) / ε_i
      sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# =========================
# Task / Job / Segment
# =========================
@dataclass(frozen=True)
class TaskBase:
    """Task parameters that do NOT change across delta_eta sweep."""
    name: str
    C_ns: int
    T_ns: int

@dataclass(frozen=True)
class TaskEps:
    """Task parameters for epsilon scheduler (delta_eta changes)."""
    name: str
    C_ns: int
    T_ns: int
    epsilon_i: float
    J_i: int
    delta_eta_i_ns: int
    T_perp_ns: int
    T_top_ns: int
    offset_ns: int = 0

    def D(self) -> int:
        return self.T_ns  # D=T

@dataclass
class Job:
    task: TaskEps
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

@dataclass
class Segment:
    start_ns: int
    end_ns: int
    task_name: str
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# =========================
# Taskset generation (fixed across delta_eta sweep)
# =========================
def generate_taskset_base(rng: random.Random) -> List[TaskBase]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_TOTAL, rng)

    tasks: List[TaskBase] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))
        tasks.append(TaskBase(name=f"tau{i}", C_ns=C_ns, T_ns=T_ns))
    return tasks  # sorted by T

def generate_one_schedulable_taskset(base_seed: int, trial_idx: int) -> Tuple[List[TaskBase], bool]:
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_base(rng)
        if not FILTER_SCHEDULABLE or rm_rta_schedulable(last):
            return last, True
    return last, False

def pick_random_pair(task_names: List[str], rng: random.Random) -> Tuple[str, str]:
    a, v = rng.sample(task_names, 2)
    return a, v

def make_eps_tasks(base_tasks: List[TaskBase], delta_eta_ms: int) -> List[TaskEps]:
    delta_eta_ns = int(delta_eta_ms * NS_PER_MS)
    return [
        TaskEps(
            name=t.name,
            C_ns=t.C_ns,
            T_ns=t.T_ns,
            epsilon_i=EPSILON,
            J_i=J,
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        )
        for t in base_tasks
    ]

# =========================
# Counting patterns + denominator
# =========================
@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

def simulate_count_patterns(
    tasks: List[TaskEps],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    victim: str,
) -> Totals:
    rng = random.Random(sim_seed)
    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()
    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    def reset_chain():
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ns: int):
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia
                else:
                    next_release[t.name] = time_ns + t.T_ns

    def push_segment(seg: Segment):
        nonlocal prev2, prev1, last

        # merge contiguous same (task, job)
        if last is not None and seg.start_ns == last.end_ns and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ns = seg.end_ns
            return

        prev2, prev1 = prev1, last
        last = seg

        if seg.task_name == attacker:
            totals.attacker_segments += 1

        if prev1 is not None and prev1.end_ns == seg.start_ns:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ns == prev1.start_ns
                and prev1.end_ns == seg.start_ns
            ):
                totals.pincer += 1

    # initial releases
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            reset_chain()
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ns=t_now,
                end_ns=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

    return totals

# =========================
# Main sweep
# =========================
def main():
    # Choose a fresh seed each run; print it so you can reproduce later if needed
    base_seed = random.SystemRandom().randint(0, 2**63 - 1)
    print(f"base_seed = {base_seed}")

    # 1) Generate fixed 200 tasksets + fixed attacker/victim per taskset
    tasksets: List[List[TaskBase]] = []
    pairs: List[Tuple[str, str]] = []

    pair_rng = random.Random(base_seed + 999)

    used = 0
    retries = 0
    while used < N_TRIALS:
        ts, ok = generate_one_schedulable_taskset(base_seed + retries * 1_000_000, used)
        if not ok:
            retries += 1
            continue
        tasksets.append(ts)
        names = [t.name for t in ts]
        pairs.append(pick_random_pair(names, pair_rng))
        used += 1

    print(f"Generated {len(tasksets)} tasksets. (schedulable_filter={FILTER_SCHEDULABLE}, retries={retries})")

    # 2) Vanilla baseline once (independent of delta_eta)
    totals_van = Totals()
    for i in range(N_TRIALS):
        base_tasks = tasksets[i]
        attacker, victim = pairs[i]

        # Vanilla uses same task struct; delta_eta doesn't matter. Use any delta_eta to build TaskEps.
        tasks_v = make_eps_tasks(base_tasks, delta_eta_ms=190)
        Tmax_ns = max(t.T_ns for t in base_tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        c = simulate_count_patterns(
            tasks=tasks_v,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=base_seed + 100_000 + i,
            attacker=attacker,
            victim=victim,
        )
        totals_van.anterior += c.anterior
        totals_van.posterior += c.posterior
        totals_van.pincer += c.pincer
        totals_van.attacker_segments += c.attacker_segments
        totals_van.attacker_jobs += c.attacker_jobs

    van_A = pct(totals_van.anterior, totals_van.attacker_segments)
    van_P = pct(totals_van.posterior, totals_van.attacker_segments)
    van_Pin = pct(totals_van.pincer, totals_van.attacker_segments)

    print("\nVanilla baseline (percent of attacker segments):")
    print(f"  Anterior={van_A:.3f}%, Posterior={van_P:.3f}%, Pincer={van_Pin:.3f}%")

    # 3) Sweep delta_eta
    rows: List[Dict[str, object]] = []
    x_vals = []
    eps_A = []
    eps_P = []
    eps_Pin = []

    for delta_eta_ms in DELTA_ETA_LIST_MS:
        totals_eps = Totals()

        for i in range(N_TRIALS):
            base_tasks = tasksets[i]
            attacker, victim = pairs[i]

            tasks_e = make_eps_tasks(base_tasks, delta_eta_ms=delta_eta_ms)
            Tmax_ns = max(t.T_ns for t in base_tasks)
            horizon_ns = HORIZON_MULT * Tmax_ns

            c = simulate_count_patterns(
                tasks=tasks_e,
                horizon_ns=horizon_ns,
                randomized=True,
                sim_seed=base_seed + 200_000 + 10_000 * delta_eta_ms + i,
                attacker=attacker,
                victim=victim,
            )
            totals_eps.anterior += c.anterior
            totals_eps.posterior += c.posterior
            totals_eps.pincer += c.pincer
            totals_eps.attacker_segments += c.attacker_segments
            totals_eps.attacker_jobs += c.attacker_jobs

        A_pct = pct(totals_eps.anterior, totals_eps.attacker_segments)
        P_pct = pct(totals_eps.posterior, totals_eps.attacker_segments)
        Pin_pct = pct(totals_eps.pincer, totals_eps.attacker_segments)

        rows.append({
            "delta_eta_ms": delta_eta_ms,
            "U_total": U_TOTAL,
            "n_tasks": N_TASKS,
            "trials": N_TRIALS,
            "epsilon": EPSILON,
            "J": J,
            "bounds_ms": f"[{T_PERP_MS},{T_TOP_MS}]",
            "horizon_mult": HORIZON_MULT,
            "vanilla_anterior_pct": van_A,
            "vanilla_posterior_pct": van_P,
            "vanilla_pincer_pct": van_Pin,
            "eps_attacker_segments": totals_eps.attacker_segments,
            "eps_attacker_jobs": totals_eps.attacker_jobs,
            "eps_anterior": totals_eps.anterior,
            "eps_posterior": totals_eps.posterior,
            "eps_pincer": totals_eps.pincer,
            "eps_anterior_pct": A_pct,
            "eps_posterior_pct": P_pct,
            "eps_pincer_pct": Pin_pct,
        })

        x_vals.append(delta_eta_ms)
        eps_A.append(A_pct)
        eps_P.append(P_pct)
        eps_Pin.append(Pin_pct)

        print(f"delta_eta={delta_eta_ms:3d} ms:  ε-RM %  A={A_pct:.3f}  P={P_pct:.3f}  Pin={Pin_pct:.3f}")

    # 4) Save CSV
    csv_path = OUT_DIR / "summary_delta_eta.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    # 5) Plot
    fig = plt.figure(figsize=(7.2, 4.2), dpi=200)
    ax = fig.add_subplot(111)

    ax.plot(x_vals, eps_A, marker="o", label="ε-RM Anterior (%)")
    ax.plot(x_vals, eps_P, marker="o", label="ε-RM Posterior (%)")
    ax.plot(x_vals, eps_Pin, marker="o", label="ε-RM Pincer (%)")

    # Vanilla as horizontal reference lines
    ax.axhline(van_A, linestyle="--", label="Vanilla Anterior (%)")
    ax.axhline(van_P, linestyle="--", label="Vanilla Posterior (%)")
    ax.axhline(van_Pin, linestyle="--", label="Vanilla Pincer (%)")

    ax.set_xlabel(r"$\Delta\eta$ (ms)")
    ax.set_ylabel("Detected instances / attacker segments (%)")
    ax.set_xticks(x_vals)
    ax.grid(True, which="both", axis="both", linestyle=":", linewidth=0.7)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()

    pdf_path = OUT_DIR / "delta_eta_sweep_plot.pdf"
    png_path = OUT_DIR / "delta_eta_sweep_plot.png"
    fig.savefig(pdf_path)
    fig.savefig(png_path)
    plt.close(fig)

    print("\nSaved:")
    print(f"  {csv_path}")
    print(f"  {pdf_path}")
    print(f"  {png_path}")

if __name__ == "__main__":
    main()

base_seed = 4383573155791836572
Generated 200 tasksets. (schedulable_filter=True, retries=0)

Vanilla baseline (percent of attacker segments):
  Anterior=6.854%, Posterior=6.891%, Pincer=2.143%
delta_eta=  0 ms:  ε-RM %  A=6.854  P=6.891  Pin=2.143
delta_eta= 20 ms:  ε-RM %  A=7.167  P=7.056  Pin=2.286
delta_eta= 40 ms:  ε-RM %  A=7.316  P=7.169  Pin=2.253
delta_eta= 60 ms:  ε-RM %  A=7.171  P=6.939  Pin=2.208
delta_eta= 80 ms:  ε-RM %  A=7.261  P=7.127  Pin=2.280
delta_eta=100 ms:  ε-RM %  A=7.182  P=7.009  Pin=2.212
delta_eta=120 ms:  ε-RM %  A=7.288  P=7.130  Pin=2.293
delta_eta=140 ms:  ε-RM %  A=7.322  P=7.119  Pin=2.259
delta_eta=160 ms:  ε-RM %  A=7.208  P=7.095  Pin=2.249
delta_eta=180 ms:  ε-RM %  A=7.068  P=6.931  Pin=2.191
delta_eta=200 ms:  ε-RM %  A=7.291  P=7.049  Pin=2.249

Saved:
  delta_eta_sweep_results\summary_delta_eta.csv
  delta_eta_sweep_results\delta_eta_sweep_plot.pdf
  delta_eta_sweep_results\delta_eta_sweep_plot.png


# control stability attacker: HP

In [49]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple

# ============================================================
# Global configuration
# ============================================================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000

# Experiment setup
N_TASKS = 10
N_TRIALS = 200
UTILIZATION_POINTS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

# Task parameters
PERIOD_MIN_MS = 20
PERIOD_MAX_MS = 200
HORIZON_MULT = 200          # simulate for HORIZON_MULT * max period
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 200

# DP / epsilon scheduler parameters
GLOBAL_EPSILON = 1000.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

GLOBAL_J = 1
FIXED_DELTA_ETA_MS = 200
DELTA_ETA_FROM_BOUNDS = False

GLOBAL_T_PERP_MS = 20
GLOBAL_T_TOP_MS = 200

T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS = GLOBAL_T_TOP_MS * NS_PER_MS

# New control-stability criterion
CONTROL_JITTER_THRESHOLD_MS = 20.83

# Output
SAVE_DIR = "control_attack_results"

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    if den == 0:
        return 0.0
    return 100.0 * num / den

def laplace_sample(rng: random.Random, mu: float, b: float) -> float:
    """
    Sample from Laplace(mu, b) using inverse CDF.
    """
    if b <= 0:
        return mu
    u = rng.random() - 0.5
    return mu - b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
) -> int:
    """
    Bounded Laplace-style inter-arrival sampler.

    If you already have your own bounded_laplace_interarrival_ns() from your
    previous script, you can drop that function body in here unchanged for
    exact continuity with earlier experiments.

    This implementation:
      - centers at mu_ns
      - uses a scale that shrinks as epsilon increases
      - truncates/clips to [t_perp_ns, t_top_ns]
    """
    eps = max(float(epsilon_i), 1e-12)
    # simple scale: larger delta_eta or J => larger spread, larger epsilon => smaller spread
    b = max(1.0, (float(J_i) * float(delta_eta_i_ns)) / eps)

    # try rejection sampling a few times
    for _ in range(100):
        x = int(round(laplace_sample(rng, mu_ns, b)))
        if t_perp_ns <= x <= t_top_ns:
            return x

    # fallback: clip
    x = int(round(laplace_sample(rng, mu_ns, b)))
    return max(t_perp_ns, min(x, t_top_ns))

def uunifast(n: int, u_total: float, rng: random.Random) -> List[float]:
    """
    UUniFast task utilization generator.
    """
    utils = []
    sum_u = u_total
    for i in range(1, n):
        next_sum_u = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum_u)
        sum_u = next_sum_u
    utils.append(sum_u)
    return utils

# ============================================================
# Task / job models
# ============================================================
@dataclass
class Task:
    name: str
    C_ns: int
    T_ns: int
    epsilon_i: float
    J_i: int
    delta_eta_i_ns: int
    T_perp_ns: int
    T_top_ns: int
    offset_ns: int = 0

    def D(self) -> int:
        return self.T_ns  # implicit deadlines

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    """
    RM priority: smaller period => higher priority.
    """
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# RM schedulability test
# ============================================================
def rm_rta_schedulable(tasks: List[Task]) -> bool:
    """
    Standard response-time analysis for implicit-deadline fixed-priority RM.
    Assumes tasks are sorted by period (increasing T_ns).
    """
    tasks_sorted = sorted(tasks, key=lambda t: t.T_ns)

    for i, ti in enumerate(tasks_sorted):
        R = ti.C_ns
        while True:
            interference = 0
            for hp in tasks_sorted[:i]:
                interference += math.ceil(R / hp.T_ns) * hp.C_ns
            R_next = ti.C_ns + interference

            if R_next == R:
                break
            if R_next > ti.D():
                return False
            R = R_next

        if R > ti.D():
            return False

    return True

# ============================================================
# Taskset generation
# ============================================================
def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)
        delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))

    return tasks

def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True
    return last, False

# ============================================================
# Trial-level result and aggregate totals
# ============================================================
@dataclass
class TrialResult:
    best_rt_ms: float = 0.0
    worst_rt_ms: float = 0.0
    jitter_ms: float = 0.0
    jobs_seen: int = 0
    failed: bool = False

@dataclass
class Totals:
    failed_tasksets: int = 0
    sum_jitter_ms: float = 0.0
    max_jitter_ms: float = 0.0
    target_jobs_total: int = 0
    sum_best_rt_ms: float = 0.0
    sum_worst_rt_ms: float = 0.0

# ============================================================
# Simulation:
# For one taskset, compute ATTACKER jitter and mark failure
# if jitter > CONTROL_JITTER_THRESHOLD_MS
# ============================================================
def simulate_attacker_jitter_failure(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[TrialResult, int, int]:
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    ia_total = 0
    ia_diff = 0

    attacker_rts_ns: List[int] = []

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid,
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            if cur.task.name == attacker:
                rt_ns = t_now - cur.release_ns
                attacker_rts_ns.append(rt_ns)
            ready.remove(cur)

    if len(attacker_rts_ns) >= 1:
        best_rt_ns = min(attacker_rts_ns)
        worst_rt_ns = max(attacker_rts_ns)
        jitter_ns = worst_rt_ns - best_rt_ns

        best_rt_ms = best_rt_ns / NS_PER_MS
        worst_rt_ms = worst_rt_ns / NS_PER_MS
        jitter_ms = jitter_ns / NS_PER_MS
    else:
        best_rt_ms = 0.0
        worst_rt_ms = 0.0
        jitter_ms = 0.0

    result = TrialResult(
        best_rt_ms=best_rt_ms,
        worst_rt_ms=worst_rt_ms,
        jitter_ms=jitter_ms,
        jobs_seen=len(attacker_rts_ns),
        failed=(jitter_ms > CONTROL_JITTER_THRESHOLD_MS),
    )

    return result, ia_total, ia_diff

# ============================================================
# One utilization run -> returns two summary rows
# ============================================================
def run_for_utilization(U_total: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object]]:
    pair_rng = random.Random(global_seed + int(U_total * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed_tasksets = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, U_total)
        if not ok:
            failed_tasksets += 1
            extra_seed_bump += 1_000_000
            continue

        # attacker = higher-priority task under RM => smaller period
        t1, t2 = pair_rng.sample(tasks, 2)
        if t1.T_ns < t2.T_ns:
            attacker = t1.name
            victim = t2.name
        else:
            attacker = t2.name
            victim = t1.name

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_res, _, _ = simulate_attacker_jitter_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker,
            collect_ia_stats=False,
        )
        totals_vs.failed_tasksets += int(vs_res.failed)
        totals_vs.sum_jitter_ms += vs_res.jitter_ms
        totals_vs.max_jitter_ms = max(totals_vs.max_jitter_ms, vs_res.jitter_ms)
        totals_vs.target_jobs_total += vs_res.jobs_seen
        totals_vs.sum_best_rt_ms += vs_res.best_rt_ms
        totals_vs.sum_worst_rt_ms += vs_res.worst_rt_ms

        # Epsilon RM
        eps_res, ia_total, ia_diff = simulate_attacker_jitter_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker,
            collect_ia_stats=True,
        )
        totals_eps.failed_tasksets += int(eps_res.failed)
        totals_eps.sum_jitter_ms += eps_res.jitter_ms
        totals_eps.max_jitter_ms = max(totals_eps.max_jitter_ms, eps_res.jitter_ms)
        totals_eps.target_jobs_total += eps_res.jobs_seen
        totals_eps.sum_best_rt_ms += eps_res.best_rt_ms
        totals_eps.sum_worst_rt_ms += eps_res.worst_rt_ms

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        used += 1

    common = {
        "utilization": U_total,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets,
        "seed": global_seed,
        "control_jitter_threshold_ms": CONTROL_JITTER_THRESHOLD_MS,
        "attacker_priority_rule": "higher_priority_than_victim",
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "taskset_failures": totals_vs.failed_tasksets,
        "taskset_failure_pct": pct(totals_vs.failed_tasksets, N_TRIALS),
        "attacker_jobs_total": totals_vs.target_jobs_total,
        "avg_jitter_ms": totals_vs.sum_jitter_ms / N_TRIALS,
        "max_jitter_ms": totals_vs.max_jitter_ms,
        "avg_best_rt_ms": totals_vs.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_vs.sum_worst_rt_ms / N_TRIALS,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={'T_top-T_perp' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "taskset_failures": totals_eps.failed_tasksets,
        "taskset_failure_pct": pct(totals_eps.failed_tasksets, N_TRIALS),
        "attacker_jobs_total": totals_eps.target_jobs_total,
        "avg_jitter_ms": totals_eps.sum_jitter_ms / N_TRIALS,
        "max_jitter_ms": totals_eps.max_jitter_ms,
        "avg_best_rt_ms": totals_eps.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_eps.sum_worst_rt_ms / N_TRIALS,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps

# ============================================================
# Main
# ============================================================
def main():
    global_seed = 1772329000000000000  # change as needed
    os.makedirs(SAVE_DIR, exist_ok=True)

    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {os.path.abspath(SAVE_DIR)}")

    all_rows: List[Dict[str, object]] = []

    for U_total in UTILIZATION_POINTS:
        print(f"\n--- Running utilization U={U_total:.1f} ---")

        row_vs, row_eps = run_for_utilization(U_total, global_seed)

        print(
            "Vanilla RM:",
            f"failures={row_vs['taskset_failures']} / {row_vs['trials']}",
            f"({row_vs['taskset_failure_pct']:.3f}%),",
            f"avg_jitter={row_vs['avg_jitter_ms']:.3f} ms,",
            f"max_jitter={row_vs['max_jitter_ms']:.3f} ms",
        )

        print(
            "Epsilon RM:",
            f"failures={row_eps['taskset_failures']} / {row_eps['trials']}",
            f"({row_eps['taskset_failure_pct']:.3f}%),",
            f"avg_jitter={row_eps['avg_jitter_ms']:.3f} ms,",
            f"max_jitter={row_eps['max_jitter_ms']:.3f} ms,",
            f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}",
        )

        all_rows.append(row_vs)
        all_rows.append(row_eps)

    summary_csv = os.path.join(SAVE_DIR, "summary_control_attack_all_utilizations.csv")
    if all_rows:
        fieldnames = list(all_rows[0].keys())
        with open(summary_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)

    print("\nDone.")
    print(f"Combined summary: {os.path.abspath(summary_csv)}")

if __name__ == "__main__":
    main()

global_seed = 1772329000000000000
Saving results to: c:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\control_attack_results

--- Running utilization U=0.1 ---
Vanilla RM: failures=0 / 200 (0.000%), avg_jitter=1.897 ms, max_jitter=9.645 ms
Epsilon RM: failures=0 / 200 (0.000%), avg_jitter=1.912 ms, max_jitter=9.645 ms, IA-diff-frac=1.0000

--- Running utilization U=0.2 ---
Vanilla RM: failures=1 / 200 (0.500%), avg_jitter=3.391 ms, max_jitter=22.466 ms
Epsilon RM: failures=1 / 200 (0.500%), avg_jitter=3.421 ms, max_jitter=22.466 ms, IA-diff-frac=1.0000

--- Running utilization U=0.3 ---
Vanilla RM: failures=9 / 200 (4.500%), avg_jitter=5.283 ms, max_jitter=30.598 ms
Epsilon RM: failures=9 / 200 (4.500%), avg_jitter=5.359 ms, max_jitter=30.598 ms, IA-diff-frac=1.0000

--- Running utilization U=0.4 ---
Vanilla RM: failures=18 / 200 (9.000%), avg_jitter=7.211 ms, max_jitter=41.668 ms
Epsilon RM: failures=18 / 200 (9.000%), 

In [48]:
# util_sweep_control_failure.py
# For each utilization U in U_LIST:
#   - generate 200 random 10-task RM tasksets (optionally filtered schedulable by RM-RTA, D=T)
#   - randomly choose two tasks from the taskset
#   - force attacker to be the HIGHER-priority one under RM (smaller period)
#   - run Vanilla RM and ε-RM
#   - compute ATTACKER response-time jitter in each taskset:
#         jitter = worst_response_time - best_response_time
#   - mark the taskset as CONTROL-STABILITY FAILURE if jitter > 20.83 ms
#   - report failure percentage over the 200 tasksets

from __future__ import annotations
from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# =========================
# Sweep settings
# =========================
U_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

N_TRIALS = 200
N_TASKS = 10

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200

# Horizon = HORIZON_MULT * Tmax (Tmax = max base period in each taskset)
HORIZON_MULT = 200

# =========================
# Control-stability threshold
# =========================
CONTROL_JITTER_THRESHOLD_MS = 20.83

# =========================
# ε-scheduler parameters (paper-style)
# =========================
GLOBAL_T_PERP_MS = 10
GLOBAL_T_TOP_MS = 200
GLOBAL_J = 16

GLOBAL_EPSILON = 1000.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# If True, Δη = (T_top - T_perp) (common: 200-10 = 190 ms)
DELTA_ETA_FROM_BOUNDS = True
FIXED_DELTA_ETA_MS = 190.0

# =========================
# Taskset filtering
# =========================
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 8000

# =========================
# Output
# =========================
OUT_DIR = Path("util_sweep_control_failure_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Time units
# =========================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000
T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS = GLOBAL_T_TOP_MS * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0

# ============================================================
# RM response-time analysis (D=T) for filtering tasksets
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]

        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns
            R_next = ti.C_ns + interference

            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next

        if R > D:
            return False

    return True

# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")
    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils

# ============================================================
# Laplace (continuous) sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5  # (-0.5, 0.5)
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Paper-style bounded Laplace:
      b = (2 * J_i * Δη_i) / ε_i
      sample cand ~ Laplace(mu, b), resample until cand in [t_perp, t_top]
    """
    if epsilon_i <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    b_ns = (2.0 * float(J_i) * float(delta_eta_i_ns)) / float(epsilon_i)
    if b_ns <= 0:
        return int(min(max(mu_ns, t_perp_ns), t_top_ns))

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b_ns)
        cand = int(round(float(mu_ns) + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    # fallback
    return int(min(max(mu_ns, t_perp_ns), t_top_ns))

# ============================================================
# Task / Job
# ============================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ns: int
    T_ns: int
    D_ns: Optional[int] = None

    # epsilon scheduler params
    epsilon_i: float = GLOBAL_EPSILON
    J_i: int = GLOBAL_J
    delta_eta_i_ns: int = int(FIXED_DELTA_ETA_MS * NS_PER_MS)

    # bounds
    T_perp_ns: int = T_PERP_NS
    T_top_ns: int = T_TOP_NS

    offset_ns: int = 0

    def D(self) -> int:
        return self.D_ns if self.D_ns is not None else self.T_ns

@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int

def rm_pick(ready: List[Job]) -> Job:
    # smaller T => higher priority
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))

# ============================================================
# Taskset generation (10 tasks, sorted by period)
# ============================================================
def generate_taskset_10_rm(rng: random.Random, U_total: float) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS

        # quantize C to microseconds to avoid pathological tiny fractions
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)
        delta_eta_ns = int(T_TOP_NS - T_PERP_NS) if DELTA_ETA_FROM_BOUNDS else int(FIXED_DELTA_ETA_MS * NS_PER_MS)

        tasks.append(Task(
            name=f"tau{i}",
            C_ns=C_ns,
            T_ns=T_ns,
            epsilon_i=eps_i,
            J_i=int(GLOBAL_J),
            delta_eta_i_ns=delta_eta_ns,
            T_perp_ns=T_PERP_NS,
            T_top_ns=T_TOP_NS,
        ))
    return tasks

def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if filtering enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last = None
    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True
    return last, False

# ============================================================
# Per-taskset result / aggregate totals
# ============================================================
@dataclass
class TrialResult:
    best_rt_ms: float = 0.0
    worst_rt_ms: float = 0.0
    jitter_ms: float = 0.0
    attacker_jobs: int = 0
    failed: bool = False

@dataclass
class Totals:
    failed_tasksets: int = 0
    attacker_jobs_total: int = 0
    sum_best_rt_ms: float = 0.0
    sum_worst_rt_ms: float = 0.0
    sum_jitter_ms: float = 0.0
    max_jitter_ms: float = 0.0

# ============================================================
# Simulation:
# For one taskset, compute ATTACKER jitter and mark failure
# if jitter > CONTROL_JITTER_THRESHOLD_MS
# ============================================================
def simulate_attacker_jitter_failure(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    attacker: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[TrialResult, int, int]:
    """
    Returns:
      TrialResult,
      ia_total_samples,
      ia_diff_from_T_samples
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    ia_total = 0
    ia_diff = 0

    attacker_rts_ns: List[int] = []

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff
        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                ready.append(Job(
                    task=t,
                    release_ns=time_ns,
                    abs_deadline_ns=time_ns + t.D(),
                    remaining_ns=t.C_ns,
                    job_id=jid
                ))

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    next_release[t.name] = time_ns + t.T_ns

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            if cur.task.name == attacker:
                rt_ns = t_now - cur.release_ns
                attacker_rts_ns.append(rt_ns)
            ready.remove(cur)

    if attacker_rts_ns:
        best_rt_ns = min(attacker_rts_ns)
        worst_rt_ns = max(attacker_rts_ns)
        jitter_ns = worst_rt_ns - best_rt_ns

        best_rt_ms = best_rt_ns / NS_PER_MS
        worst_rt_ms = worst_rt_ns / NS_PER_MS
        jitter_ms = jitter_ns / NS_PER_MS
    else:
        best_rt_ms = 0.0
        worst_rt_ms = 0.0
        jitter_ms = 0.0

    result = TrialResult(
        best_rt_ms=best_rt_ms,
        worst_rt_ms=worst_rt_ms,
        jitter_ms=jitter_ms,
        attacker_jobs=len(attacker_rts_ns),
        failed=(jitter_ms > CONTROL_JITTER_THRESHOLD_MS),
    )

    return result, ia_total, ia_diff

# ============================================================
# One utilization run -> returns two summary rows + trial details
# ============================================================
def run_for_utilization(U_total: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object], List[Dict[str, object]]]:
    pair_rng = random.Random(global_seed + int(U_total * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    used = 0
    failed_tasksets_generation = 0
    extra_seed_bump = 0

    trial_rows: List[Dict[str, object]] = []

    while used < N_TRIALS:
        tasks, ok = generate_taskset_filtered(global_seed + extra_seed_bump, used, U_total)
        if not ok:
            failed_tasksets_generation += 1
            extra_seed_bump += 1_000_000
            continue

        # choose a random pair, but force ATTACKER = higher-priority task under RM
        t1, t2 = pair_rng.sample(tasks, 2)
        if t1.T_ns < t2.T_ns:
            attacker = t1.name
            victim = t2.name
        else:
            attacker = t2.name
            victim = t1.name

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_res, _, _ = simulate_attacker_jitter_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            attacker=attacker,
            collect_ia_stats=False,
        )
        totals_vs.failed_tasksets += int(vs_res.failed)
        totals_vs.attacker_jobs_total += vs_res.attacker_jobs
        totals_vs.sum_best_rt_ms += vs_res.best_rt_ms
        totals_vs.sum_worst_rt_ms += vs_res.worst_rt_ms
        totals_vs.sum_jitter_ms += vs_res.jitter_ms
        totals_vs.max_jitter_ms = max(totals_vs.max_jitter_ms, vs_res.jitter_ms)

        # ε-RM
        eps_res, ia_total, ia_diff = simulate_attacker_jitter_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            attacker=attacker,
            collect_ia_stats=True,
        )
        totals_eps.failed_tasksets += int(eps_res.failed)
        totals_eps.attacker_jobs_total += eps_res.attacker_jobs
        totals_eps.sum_best_rt_ms += eps_res.best_rt_ms
        totals_eps.sum_worst_rt_ms += eps_res.worst_rt_ms
        totals_eps.sum_jitter_ms += eps_res.jitter_ms
        totals_eps.max_jitter_ms = max(totals_eps.max_jitter_ms, eps_res.jitter_ms)

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        trial_rows.append({
            "utilization": U_total,
            "trial_index": used,
            "attacker": attacker,
            "victim": victim,
            "attacker_period_ms": next(t.T_ns for t in tasks if t.name == attacker) / NS_PER_MS,
            "victim_period_ms": next(t.T_ns for t in tasks if t.name == victim) / NS_PER_MS,

            "vanilla_best_rt_ms": vs_res.best_rt_ms,
            "vanilla_worst_rt_ms": vs_res.worst_rt_ms,
            "vanilla_jitter_ms": vs_res.jitter_ms,
            "vanilla_failed": int(vs_res.failed),

            "epsilon_best_rt_ms": eps_res.best_rt_ms,
            "epsilon_worst_rt_ms": eps_res.worst_rt_ms,
            "epsilon_jitter_ms": eps_res.jitter_ms,
            "epsilon_failed": int(eps_res.failed),
        })

        used += 1

    common = {
        "utilization": U_total,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets_generation,
        "seed": global_seed,
        "control_jitter_threshold_ms": CONTROL_JITTER_THRESHOLD_MS,
        "attacker_priority_rule": "higher_priority_than_victim",
    }

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "taskset_failures": totals_vs.failed_tasksets,
        "taskset_failure_pct": pct(totals_vs.failed_tasksets, N_TRIALS),
        "attacker_jobs_total": totals_vs.attacker_jobs_total,
        "avg_best_rt_ms": totals_vs.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_vs.sum_worst_rt_ms / N_TRIALS,
        "avg_jitter_ms": totals_vs.sum_jitter_ms / N_TRIALS,
        "max_jitter_ms": totals_vs.max_jitter_ms,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={'T_top-T_perp' if DELTA_ETA_FROM_BOUNDS else str(FIXED_DELTA_ETA_MS)+'ms'}, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "taskset_failures": totals_eps.failed_tasksets,
        "taskset_failure_pct": pct(totals_eps.failed_tasksets, N_TRIALS),
        "attacker_jobs_total": totals_eps.attacker_jobs_total,
        "avg_best_rt_ms": totals_eps.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_eps.sum_worst_rt_ms / N_TRIALS,
        "avg_jitter_ms": totals_eps.sum_jitter_ms / N_TRIALS,
        "max_jitter_ms": totals_eps.max_jitter_ms,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps, trial_rows

# ============================================================
# Main
# ============================================================
def main():
    global_seed = int(time.time_ns())

    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    all_summary_rows: List[Dict[str, object]] = []

    for U_total in U_LIST:
        print(f"\n--- Running utilization U={U_total:.1f} ---")

        row_vs, row_eps, trial_rows = run_for_utilization(U_total, global_seed)

        # console output
        print(
            "Vanilla RM:",
            f"failures={row_vs['taskset_failures']} ({row_vs['taskset_failure_pct']:.3f}%),",
            f"avg_jitter={row_vs['avg_jitter_ms']:.3f} ms,",
            f"max_jitter={row_vs['max_jitter_ms']:.3f} ms"
        )

        print(
            "Epsilon RM:",
            f"failures={row_eps['taskset_failures']} ({row_eps['taskset_failure_pct']:.3f}%),",
            f"avg_jitter={row_eps['avg_jitter_ms']:.3f} ms,",
            f"max_jitter={row_eps['max_jitter_ms']:.3f} ms",
            f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}"
        )

        # save per-utilization trial details
        details_csv = OUT_DIR / f"control_failure_trials_U_{str(U_total).replace('.', 'p')}.csv"
        if trial_rows:
            with open(details_csv, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=list(trial_rows[0].keys()))
                writer.writeheader()
                writer.writerows(trial_rows)

        # save per-utilization summary (2 rows: Vanilla / Epsilon)
        util_summary_csv = OUT_DIR / f"summary_U_{str(U_total).replace('.', 'p')}.csv"
        util_rows = [row_vs, row_eps]
        with open(util_summary_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(util_rows[0].keys()))
            writer.writeheader()
            writer.writerows(util_rows)

        all_summary_rows.extend(util_rows)

    combined_csv = OUT_DIR / "summary_all_utilizations.csv"
    if all_summary_rows:
        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_summary_rows[0].keys()))
            writer.writeheader()
            writer.writerows(all_summary_rows)

    print("\nDone.")
    print(f"Combined summary: {combined_csv.resolve()}")

if __name__ == "__main__":
    main()

global_seed = 1772392239270262900
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\util_sweep_control_failure_results

--- Running utilization U=0.1 ---
Vanilla RM: failures=0 (0.000%), avg_jitter=1.337 ms, max_jitter=9.712 ms
Epsilon RM: failures=0 (0.000%), avg_jitter=1.342 ms, max_jitter=9.712 ms IA-diff-frac=1.0000

--- Running utilization U=0.2 ---
Vanilla RM: failures=0 (0.000%), avg_jitter=2.794 ms, max_jitter=17.723 ms
Epsilon RM: failures=0 (0.000%), avg_jitter=2.829 ms, max_jitter=18.180 ms IA-diff-frac=1.0000

--- Running utilization U=0.3 ---
Vanilla RM: failures=5 (2.500%), avg_jitter=4.714 ms, max_jitter=29.581 ms
Epsilon RM: failures=5 (2.500%), avg_jitter=4.768 ms, max_jitter=29.581 ms IA-diff-frac=1.0000

--- Running utilization U=0.4 ---
Vanilla RM: failures=13 (6.500%), avg_jitter=6.019 ms, max_jitter=36.663 ms
Epsilon RM: failures=13 (6.500%), avg_jitter=6.105 ms, max_jitter=36.752 

# vary delta_eta, Butterfly

In [1]:
# delta_eta_sweep_control_failure.py
# ------------------------------------------------------------
# For each delta_eta in {0,20,40,...,200} ms:
#   - generate N_TRIALS random RM tasksets at fixed utilization U_TOTAL
#   - choose two tasks from the taskset
#   - by default, force attacker to be higher priority than victim under RM
#   - use the victim as the control task
#   - run Vanilla RM (fixed period) and epsilon-RM (randomized inter-arrivals)
#   - compute control-task response-time spread:
#         L = best response time
#         J = worst response time - best response time
#   - mark control-stability failure by either:
#         (A) J > CONTROL_JITTER_THRESHOLD_MS
#       or
#         (B) L + ALPHA * J > BETA
#   - save summary CSV + per-trial CSV
# ------------------------------------------------------------

from __future__ import annotations

from dataclasses import dataclass
import csv
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ============================================================
# Sweep settings
# ============================================================
DELTA_ETA_LIST_MS = list(range(0, 201, 20))   # 0,20,...,200
U_TOTAL = 0.3                                 # fixed utilization for all runs

N_TRIALS = 500
N_TASKS = 10

PERIOD_MIN_MS = 10
PERIOD_MAX_MS = 200

# Horizon = HORIZON_MULT * Tmax, where Tmax is max base period in the taskset
HORIZON_MULT = 200

# ============================================================
# Failure criterion
# ============================================================
# Choose one:
#   "jitter_threshold"  -> fail if J > CONTROL_JITTER_THRESHOLD_MS
#   "L_alpha_J_beta"    -> fail if L + ALPHA * J > BETA
FAIL_MODE = "jitter_threshold"

CONTROL_JITTER_THRESHOLD_MS = 20.83

# If FAIL_MODE == "L_alpha_J_beta", use these:
ALPHA = 1.2
BETA = 20.83

# ============================================================
# Attacker / victim selection
# ============================================================
# If True, attacker is forced to be higher priority than victim under RM
# (smaller period => higher priority).
# If False, attacker/victim roles are assigned randomly.
FORCE_ATTACKER_HIGHER_PRIORITY = True

# ============================================================
# epsilon-scheduler parameters
# ============================================================
GLOBAL_T_PERP_MS = 10
GLOBAL_T_TOP_MS = 200
GLOBAL_J = 16

GLOBAL_EPSILON = 1.0
RANDOMIZE_EPSILON_PER_TASK = False
EPSILON_CHOICES = [10.0, 100.0, 1000.0]

# For this script, delta_eta is swept explicitly, so keep this False.
DELTA_ETA_FROM_BOUNDS = False

# ============================================================
# Taskset filtering
# ============================================================
FILTER_SCHEDULABLE = True
MAX_GEN_ATTEMPTS = 8000

# ============================================================
# Output
# ============================================================
OUT_DIR = Path("delta_eta_sweep_control_failure_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Time units
# ============================================================
NS_PER_US = 1_000
NS_PER_MS = 1_000_000

T_PERP_NS = GLOBAL_T_PERP_MS * NS_PER_MS
T_TOP_NS = GLOBAL_T_TOP_MS * NS_PER_MS

# ============================================================
# Helpers
# ============================================================
def pct(num: int, den: int) -> float:
    return (100.0 * num / den) if den > 0 else 0.0


# ============================================================
# RM response-time analysis (D = T) for taskset filtering
# ============================================================
def rm_rta_schedulable(tasks_sorted_by_T: List["Task"]) -> bool:
    for i, ti in enumerate(tasks_sorted_by_T):
        D = ti.D()
        R = ti.C_ns
        hp = tasks_sorted_by_T[:i]

        for _ in range(10_000):
            interference = 0
            for tj in hp:
                interference += ((R + tj.T_ns - 1) // tj.T_ns) * tj.C_ns

            R_next = ti.C_ns + interference

            if R_next == R:
                break
            if R_next > D:
                return False
            R = R_next

        if R > D:
            return False

    return True


# ============================================================
# UUniFast utilization generation
# ============================================================
def uunifast(n: int, U_total: float, rng: random.Random) -> List[float]:
    if not (0.0 < U_total < 1.0):
        raise ValueError("U_total must be in (0,1)")

    utils = []
    sum_u = U_total
    for i in range(1, n):
        next_sum = sum_u * (rng.random() ** (1.0 / (n - i)))
        utils.append(sum_u - next_sum)
        sum_u = next_sum
    utils.append(sum_u)
    return utils


# ============================================================
# Laplace sampler + bounded resampling
# ============================================================
def laplace_sample_0_b(rng: random.Random, b: float) -> float:
    """
    Sample Laplace(0, b) via inverse CDF.
    """
    if b <= 0.0:
        return 0.0
    u = rng.random() - 0.5  # (-0.5, 0.5)
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))


def bounded_laplace_interarrival_ns(
    rng: random.Random,
    mu_ns: int,
    epsilon_i: float,
    J_i: int,
    delta_eta_i_ns: int,
    t_perp_ns: int,
    t_top_ns: int,
    *,
    max_tries: int = 50_000,
) -> int:
    """
    Sample an inter-arrival time around mu_ns using a bounded Laplace mechanism.

    Scale:
        b = 2 * J_i * delta_eta / epsilon_i

    Then resample until the result lies inside [t_perp_ns, t_top_ns].
    """

    if epsilon_i <= 0:
        raise ValueError("epsilon_i must be > 0")
    if t_perp_ns > t_top_ns:
        raise ValueError("t_perp_ns must be <= t_top_ns")
    if delta_eta_i_ns < 0:
        raise ValueError("delta_eta_i_ns must be >= 0")

    # If delta_eta = 0, the mechanism degenerates to the mean period.
    if delta_eta_i_ns == 0:
        return max(t_perp_ns, min(mu_ns, t_top_ns))

    b = (2.0 * J_i * float(delta_eta_i_ns)) / float(epsilon_i)

    for _ in range(max_tries):
        noise = laplace_sample_0_b(rng, b)
        cand = int(round(mu_ns + noise))
        if t_perp_ns <= cand <= t_top_ns:
            return cand

    # Fallback if resampling failed too long:
    return max(t_perp_ns, min(mu_ns, t_top_ns))


# ============================================================
# Task / Job
# ============================================================
@dataclass
class Task:
    name: str
    C_ns: int
    T_ns: int

    epsilon_i: float
    J_i: int
    delta_eta_i_ns: int
    T_perp_ns: int
    T_top_ns: int

    offset_ns: int = 0

    def D(self) -> int:
        return self.T_ns


@dataclass
class Job:
    task: Task
    release_ns: int
    abs_deadline_ns: int
    remaining_ns: int
    job_id: int


def rm_pick(ready: List[Job]) -> Job:
    # Smaller T => higher priority
    return min(ready, key=lambda j: (j.task.T_ns, j.abs_deadline_ns, j.release_ns, j.job_id))


# ============================================================
# Taskset generation
# ============================================================
def generate_taskset_10_rm(
    rng: random.Random,
    U_total: float,
    delta_eta_ms: float,
) -> List[Task]:
    periods_ms = sorted(rng.sample(range(PERIOD_MIN_MS, PERIOD_MAX_MS + 1), N_TASKS))
    utils = uunifast(N_TASKS, U_total, rng)

    tasks: List[Task] = []
    for i, (Tms, u) in enumerate(zip(periods_ms, utils), start=1):
        T_ns = Tms * NS_PER_MS

        # Quantize C to microseconds to avoid tiny fractions
        C_ns = int(round(u * T_ns / NS_PER_US)) * NS_PER_US
        C_ns = max(NS_PER_US, min(C_ns, T_ns - NS_PER_US))

        eps_i = float(rng.choice(EPSILON_CHOICES)) if RANDOMIZE_EPSILON_PER_TASK else float(GLOBAL_EPSILON)

        if DELTA_ETA_FROM_BOUNDS:
            delta_eta_ns = int(T_TOP_NS - T_PERP_NS)
        else:
            delta_eta_ns = int(delta_eta_ms * NS_PER_MS)

        tasks.append(
            Task(
                name=f"tau{i}",
                C_ns=C_ns,
                T_ns=T_ns,
                epsilon_i=eps_i,
                J_i=int(GLOBAL_J),
                delta_eta_i_ns=delta_eta_ns,
                T_perp_ns=T_PERP_NS,
                T_top_ns=T_TOP_NS,
            )
        )

    return tasks


def generate_taskset_filtered(
    base_seed: int,
    trial_idx: int,
    U_total: float,
    delta_eta_ms: float,
) -> Tuple[List[Task], bool]:
    """
    Try up to MAX_GEN_ATTEMPTS to get a schedulable taskset (if filtering enabled).
    Returns (tasks, ok).
    """
    rng = random.Random(base_seed + 10_000 * trial_idx)
    last: Optional[List[Task]] = None

    for _ in range(MAX_GEN_ATTEMPTS):
        last = generate_taskset_10_rm(rng, U_total, delta_eta_ms)
        if not FILTER_SCHEDULABLE:
            return last, True
        if rm_rta_schedulable(last):
            return last, True

    return last if last is not None else [], False


# ============================================================
# Per-taskset result / aggregate totals
# ============================================================
@dataclass
class TrialResult:
    best_rt_ms: float = 0.0
    worst_rt_ms: float = 0.0
    jitter_ms: float = 0.0
    control_jobs: int = 0
    stability_metric: float = 0.0
    failed: bool = False


@dataclass
class Totals:
    failed_tasksets: int = 0
    control_jobs_total: int = 0
    sum_best_rt_ms: float = 0.0
    sum_worst_rt_ms: float = 0.0
    sum_jitter_ms: float = 0.0
    sum_stability_metric: float = 0.0
    max_jitter_ms: float = 0.0


# ============================================================
# Failure decision
# ============================================================
def is_control_failure(best_rt_ms: float, jitter_ms: float) -> Tuple[bool, float]:
    """
    Returns:
      failed, stability_metric

    If FAIL_MODE == "jitter_threshold":
        stability_metric = jitter_ms
        failed = jitter_ms > CONTROL_JITTER_THRESHOLD_MS

    If FAIL_MODE == "L_alpha_J_beta":
        stability_metric = L + alpha * J
        failed = (L + alpha * J) > beta
    """
    if FAIL_MODE == "jitter_threshold":
        metric = jitter_ms
        failed = metric > CONTROL_JITTER_THRESHOLD_MS
        return failed, metric

    if FAIL_MODE == "L_alpha_J_beta":
        metric = best_rt_ms + ALPHA * jitter_ms
        failed = metric > BETA
        return failed, metric

    raise ValueError(f"Unknown FAIL_MODE: {FAIL_MODE}")


# ============================================================
# Simulation:
# For one taskset, compute CONTROL-task RT spread and mark failure
# ============================================================
def simulate_control_failure(
    tasks: List[Task],
    horizon_ns: int,
    randomized: bool,
    sim_seed: int,
    control_task: str,
    *,
    collect_ia_stats: bool = False,
) -> Tuple[TrialResult, int, int]:
    """
    Returns:
      TrialResult,
      ia_total_samples,
      ia_diff_from_T_samples
    """
    rng = random.Random(sim_seed)

    next_release: Dict[str, int] = {t.name: t.offset_ns for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    ia_total = 0
    ia_diff = 0

    control_rts_ns: List[int] = []

    def release_at(time_ns: int):
        nonlocal ia_total, ia_diff

        for t in tasks:
            if next_release[t.name] == time_ns:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                ready.append(
                    Job(
                        task=t,
                        release_ns=time_ns,
                        abs_deadline_ns=time_ns + t.D(),
                        remaining_ns=t.C_ns,
                        job_id=jid,
                    )
                )

                if randomized:
                    ia = bounded_laplace_interarrival_ns(
                        rng=rng,
                        mu_ns=t.T_ns,
                        epsilon_i=t.epsilon_i,
                        J_i=t.J_i,
                        delta_eta_i_ns=t.delta_eta_i_ns,
                        t_perp_ns=t.T_perp_ns,
                        t_top_ns=t.T_top_ns,
                    )
                    next_release[t.name] = time_ns + ia

                    if collect_ia_stats:
                        ia_total += 1
                        if ia != t.T_ns:
                            ia_diff += 1
                else:
                    # Vanilla RM uses fixed period
                    next_release[t.name] = time_ns + t.T_ns

    # Initial releases at t = 0
    t_now = 0
    release_at(0)

    while t_now < horizon_ns:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ns:
                break
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ns
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ns)

        ran = t_event - t_now
        if ran > 0:
            cur.remaining_ns -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ns == 0:
            ready.remove(cur)

            if cur.task.name == control_task:
                rt_ns = t_now - cur.release_ns
                control_rts_ns.append(rt_ns)

    if control_rts_ns:
        best_rt_ms = min(control_rts_ns) / NS_PER_MS
        worst_rt_ms = max(control_rts_ns) / NS_PER_MS
        jitter_ms = worst_rt_ms - best_rt_ms
        failed, metric = is_control_failure(best_rt_ms, jitter_ms)

        res = TrialResult(
            best_rt_ms=best_rt_ms,
            worst_rt_ms=worst_rt_ms,
            jitter_ms=jitter_ms,
            control_jobs=len(control_rts_ns),
            stability_metric=metric,
            failed=failed,
        )
    else:
        # Extremely unlikely with synchronous release + long horizon,
        # but handle safely.
        res = TrialResult(
            best_rt_ms=0.0,
            worst_rt_ms=0.0,
            jitter_ms=0.0,
            control_jobs=0,
            stability_metric=0.0,
            failed=False,
        )

    return res, ia_total, ia_diff


# ============================================================
# One delta-eta run -> returns summary rows + per-trial rows
# ============================================================
def run_for_delta_eta(delta_eta_ms: float, global_seed: int) -> Tuple[Dict[str, object], Dict[str, object], List[Dict[str, object]]]:
    pair_rng = random.Random(global_seed + int(delta_eta_ms * 1_000_000) + 999)

    totals_vs = Totals()
    totals_eps = Totals()

    eps_ia_total_all = 0
    eps_ia_diff_all = 0

    trial_rows: List[Dict[str, object]] = []

    used = 0
    failed_tasksets_generation = 0
    extra_seed_bump = 0

    while used < N_TRIALS:
        tasks, ok = generate_taskset_filtered(
            base_seed=global_seed + extra_seed_bump,
            trial_idx=used,
            U_total=U_TOTAL,
            delta_eta_ms=delta_eta_ms,
        )

        if not ok:
            failed_tasksets_generation += 1
            extra_seed_bump += 1_000_000
            continue

        # Pick attacker / victim
        if FORCE_ATTACKER_HIGHER_PRIORITY:
            t1, t2 = pair_rng.sample(tasks, 2)
            if t1.T_ns < t2.T_ns:
                attacker = t1.name
                victim = t2.name
            else:
                attacker = t2.name
                victim = t1.name
        else:
            attacker, victim = pair_rng.sample([t.name for t in tasks], 2)

        # Use victim as the control task
        control_task = victim

        Tmax_ns = max(t.T_ns for t in tasks)
        horizon_ns = HORIZON_MULT * Tmax_ns

        # Vanilla RM
        vs_res, _, _ = simulate_control_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=False,
            sim_seed=global_seed + 100_000 + used,
            control_task=control_task,
            collect_ia_stats=False,
        )

        totals_vs.failed_tasksets += int(vs_res.failed)
        totals_vs.control_jobs_total += vs_res.control_jobs
        totals_vs.sum_best_rt_ms += vs_res.best_rt_ms
        totals_vs.sum_worst_rt_ms += vs_res.worst_rt_ms
        totals_vs.sum_jitter_ms += vs_res.jitter_ms
        totals_vs.sum_stability_metric += vs_res.stability_metric
        totals_vs.max_jitter_ms = max(totals_vs.max_jitter_ms, vs_res.jitter_ms)

        # epsilon-RM
        eps_res, ia_total, ia_diff = simulate_control_failure(
            tasks=tasks,
            horizon_ns=horizon_ns,
            randomized=True,
            sim_seed=global_seed + 200_000 + used,
            control_task=control_task,
            collect_ia_stats=True,
        )

        totals_eps.failed_tasksets += int(eps_res.failed)
        totals_eps.control_jobs_total += eps_res.control_jobs
        totals_eps.sum_best_rt_ms += eps_res.best_rt_ms
        totals_eps.sum_worst_rt_ms += eps_res.worst_rt_ms
        totals_eps.sum_jitter_ms += eps_res.jitter_ms
        totals_eps.sum_stability_metric += eps_res.stability_metric
        totals_eps.max_jitter_ms = max(totals_eps.max_jitter_ms, eps_res.jitter_ms)

        eps_ia_total_all += ia_total
        eps_ia_diff_all += ia_diff

        attacker_period_ms = next(t.T_ns for t in tasks if t.name == attacker) / NS_PER_MS
        victim_period_ms = next(t.T_ns for t in tasks if t.name == victim) / NS_PER_MS

        trial_rows.append(
            {
                "delta_eta_ms": delta_eta_ms,
                "trial_index": used,

                "attacker": attacker,
                "victim": victim,
                "control_task": control_task,

                "attacker_period_ms": attacker_period_ms,
                "victim_period_ms": victim_period_ms,

                "vanilla_best_rt_ms": vs_res.best_rt_ms,
                "vanilla_worst_rt_ms": vs_res.worst_rt_ms,
                "vanilla_jitter_ms": vs_res.jitter_ms,
                "vanilla_stability_metric": vs_res.stability_metric,
                "vanilla_failed": int(vs_res.failed),

                "epsilon_best_rt_ms": eps_res.best_rt_ms,
                "epsilon_worst_rt_ms": eps_res.worst_rt_ms,
                "epsilon_jitter_ms": eps_res.jitter_ms,
                "epsilon_stability_metric": eps_res.stability_metric,
                "epsilon_failed": int(eps_res.failed),
            }
        )

        used += 1

    common = {
        "delta_eta_ms": delta_eta_ms,
        "trials": N_TRIALS,
        "tasks_per_set": N_TASKS,
        "utilization": U_TOTAL,
        "period_min_ms": PERIOD_MIN_MS,
        "period_max_ms": PERIOD_MAX_MS,
        "horizon_mult": HORIZON_MULT,
        "schedulability_filter": FILTER_SCHEDULABLE,
        "failed_generation_retries": failed_tasksets_generation,
        "seed": global_seed,
        "fail_mode": FAIL_MODE,
        "attacker_priority_rule": "higher_priority_than_victim" if FORCE_ATTACKER_HIGHER_PRIORITY else "random_role_assignment",
    }

    if FAIL_MODE == "jitter_threshold":
        common["control_threshold"] = CONTROL_JITTER_THRESHOLD_MS
    else:
        common["control_threshold"] = f"L + {ALPHA}*J > {BETA}"

    row_vs = {
        **common,
        "scheduler": "Vanilla_RM",
        "taskset_failures": totals_vs.failed_tasksets,
        "taskset_failure_pct": pct(totals_vs.failed_tasksets, N_TRIALS),
        "control_jobs_total": totals_vs.control_jobs_total,
        "avg_best_rt_ms": totals_vs.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_vs.sum_worst_rt_ms / N_TRIALS,
        "avg_jitter_ms": totals_vs.sum_jitter_ms / N_TRIALS,
        "avg_stability_metric": totals_vs.sum_stability_metric / N_TRIALS,
        "max_jitter_ms": totals_vs.max_jitter_ms,
        "eps_params": "",
        "eps_interarrival_total": "",
        "eps_interarrival_diff": "",
        "eps_interarrival_frac_diff": "",
    }

    frac_diff = (eps_ia_diff_all / eps_ia_total_all) if eps_ia_total_all else 0.0
    eps_params_str = (
        f"eps={GLOBAL_EPSILON}(rand={RANDOMIZE_EPSILON_PER_TASK}), "
        f"J={GLOBAL_J}, "
        f"delta_eta={delta_eta_ms}ms, "
        f"bounds=[{GLOBAL_T_PERP_MS},{GLOBAL_T_TOP_MS}]ms"
    )

    row_eps = {
        **common,
        "scheduler": "Epsilon_RM",
        "taskset_failures": totals_eps.failed_tasksets,
        "taskset_failure_pct": pct(totals_eps.failed_tasksets, N_TRIALS),
        "control_jobs_total": totals_eps.control_jobs_total,
        "avg_best_rt_ms": totals_eps.sum_best_rt_ms / N_TRIALS,
        "avg_worst_rt_ms": totals_eps.sum_worst_rt_ms / N_TRIALS,
        "avg_jitter_ms": totals_eps.sum_jitter_ms / N_TRIALS,
        "avg_stability_metric": totals_eps.sum_stability_metric / N_TRIALS,
        "max_jitter_ms": totals_eps.max_jitter_ms,
        "eps_params": eps_params_str,
        "eps_interarrival_total": eps_ia_total_all,
        "eps_interarrival_diff": eps_ia_diff_all,
        "eps_interarrival_frac_diff": frac_diff,
    }

    return row_vs, row_eps, trial_rows


# ============================================================
# CSV helper
# ============================================================
def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


# ============================================================
# Main
# ============================================================
def main():
    global_seed = int(time.time_ns())
    print(f"global_seed = {global_seed}")
    print(f"Saving results to: {OUT_DIR.resolve()}")

    summary_rows: List[Dict[str, object]] = []
    trial_rows_all: List[Dict[str, object]] = []

    for delta_eta_ms in DELTA_ETA_LIST_MS:
        print(f"\n--- Running delta_eta={delta_eta_ms} ms ---")

        row_vs, row_eps, trial_rows = run_for_delta_eta(delta_eta_ms, global_seed)

        summary_rows.append(row_vs)
        summary_rows.append(row_eps)
        trial_rows_all.extend(trial_rows)

        print(
            f"Vanilla RM: failures={row_vs['taskset_failures']} "
            f"({row_vs['taskset_failure_pct']:.3f}%), "
            f"avgJ={row_vs['avg_jitter_ms']:.3f} ms, "
            f"maxJ={row_vs['max_jitter_ms']:.3f} ms"
        )

        print(
            f"Epsilon RM: failures={row_eps['taskset_failures']} "
            f"({row_eps['taskset_failure_pct']:.3f}%), "
            f"avgJ={row_eps['avg_jitter_ms']:.3f} ms, "
            f"maxJ={row_eps['max_jitter_ms']:.3f} ms, "
            f"IA-diff-frac={row_eps['eps_interarrival_frac_diff']:.4f}"
        )

    summary_path = OUT_DIR / "summary_all_delta_eta.csv"
    trial_path = OUT_DIR / "trial_details_all_delta_eta.csv"

    write_csv(summary_path, summary_rows)
    write_csv(trial_path, trial_rows_all)

    print("\nDone.")
    print(f"Combined summary: {summary_path}")
    print(f"Per-trial details: {trial_path}")


if __name__ == "__main__":
    main()

global_seed = 1773426981102616300
Saving results to: C:\Users\fakhr\OneDrive - Washington State University (email.wsu.edu)\Desktop\butterfly attack\Code folder\new exp\delta_eta_sweep_control_failure_results

--- Running delta_eta=0 ms ---
Vanilla RM: failures=154 (30.800%), avgJ=15.032 ms, maxJ=39.463 ms
Epsilon RM: failures=154 (30.800%), avgJ=15.032 ms, maxJ=39.463 ms, IA-diff-frac=0.0000

--- Running delta_eta=20 ms ---
Vanilla RM: failures=135 (27.000%), avgJ=14.814 ms, maxJ=42.174 ms
Epsilon RM: failures=171 (34.200%), avgJ=17.405 ms, maxJ=111.261 ms, IA-diff-frac=1.0000

--- Running delta_eta=40 ms ---
Vanilla RM: failures=156 (31.200%), avgJ=15.250 ms, maxJ=41.639 ms
Epsilon RM: failures=188 (37.600%), avgJ=18.237 ms, maxJ=82.163 ms, IA-diff-frac=1.0000

--- Running delta_eta=60 ms ---
Vanilla RM: failures=144 (28.800%), avgJ=15.505 ms, maxJ=44.312 ms
Epsilon RM: failures=178 (35.600%), avgJ=18.466 ms, maxJ=81.902 ms, IA-diff-frac=1.0000

--- Running delta_eta=80 ms ---
Vanilla

# trace count and anterior posterior and pincer count

In [154]:
# trace_count_rm_eps_random.py
# Self-contained working code (random every run by default):
# - Generates a random taskset (n tasks, total utilization U)
# - Computes HP (LCM of periods) and runs for N windows of length HP (or capped window length)
# - Counts unique traces per window for Vanilla RM vs ε-RM (DPS)
# - Counts Anterior/Posterior/Pincer totals over the combined horizon
#
# Run:
#   python trace_count_rm_eps_random.py
# Optional reproducibility:
#   python trace_count_rm_eps_random.py 123456789   (uses that seed)

from __future__ import annotations

import math
import random
import secrets
import sys
from dataclasses import dataclass
from collections import Counter
from typing import List, Dict, Tuple, Optional, Iterable


# =========================
# Utilities
# =========================

def lcm(a: int, b: int) -> int:
    return abs(a // math.gcd(a, b) * b)

def lcm_list(xs: Iterable[int]) -> int:
    hp = 1
    for x in xs:
        hp = lcm(hp, int(x))
    return int(hp)

def laplace_noise(rng: random.Random, b: float) -> float:
    """Sample Laplace(0,b) via inverse CDF."""
    u = rng.random() - 0.5  # U ~ (-0.5, 0.5)
    if u == 0.0:
        return 0.0
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ms(
    rng: random.Random,
    mu_ms: int,
    epsilon: float,
    J: float,
    delta_eta_ms: int,
    t_perp_ms: int,
    t_top_ms: int,
) -> int:
    """
    b = 2*J*Δη/ε, sample mu + Lap(0,b), clamp to [T_perp, T_top].
    """
    if epsilon <= 0:
        raise ValueError("epsilon must be > 0")
    b = (2.0 * float(J) * float(delta_eta_ms)) / float(epsilon)
    x = float(mu_ms) + laplace_noise(rng, b)
    ia = int(round(x))
    ia = max(1, ia)
    ia = max(t_perp_ms, min(ia, t_top_ms))
    return ia


# =========================
# Task / Job / Segment
# =========================

@dataclass(frozen=True)
class Task:
    name: str
    C_ms: int
    T_ms: int
    D_ms: Optional[int] = None
    offset_ms: int = 0

    # ε-scheduler parameters
    epsilon_i: float = 100.0
    J_i: float = 16.0
    delta_eta_ms: int = 100

    # bounds for DPS inter-arrival
    T_perp_ms: Optional[int] = None
    T_top_ms: Optional[int] = None

    def D(self) -> int:
        return self.D_ms if self.D_ms is not None else self.T_ms

@dataclass
class Job:
    task: Task
    release_ms: int
    abs_deadline_ms: int
    remaining_ms: int
    job_id: int

@dataclass
class Segment:
    start_ms: int
    end_ms: int
    task_name: str
    job_id: int

@dataclass
class Totals:
    anterior: int = 0
    posterior: int = 0
    pincer: int = 0
    attacker_segments: int = 0
    attacker_jobs: int = 0

@dataclass
class TraceResult:
    hp_used_ms: int
    hp_true_ms: int
    hp_was_capped: bool
    num_windows: int
    unique_traces: int
    all_windows_match: bool
    trace_freq: Counter
    totals: Totals


# =========================
# RM priority selection
# =========================

def rm_pick(ready: List[Job]) -> Job:
    # Smaller T => higher priority. Deterministic tie-break.
    return min(ready, key=lambda j: (j.task.T_ms, j.release_ms, j.task.name, j.job_id))


# =========================
# Taskset generation
# =========================

def generate_taskset(
    *,
    n_tasks: int,
    U_total: float,
    seed: int,
    period_pool_ms: Optional[List[int]] = None,
    period_min_ms: int = 50,
    period_max_ms: int = 150,
    epsilon: float = 1.0,
    J: float = 16.0,
    delta_eta_ms: int = 100,
) -> List[Task]:
    """
    Generates a taskset with D=T, offsets=0, utilization sum ≈ U_total.

    If period_pool_ms is provided, periods are drawn from that pool (keeps HP manageable).
    Otherwise periods are drawn uniformly from [period_min_ms, period_max_ms] (HP may explode).
    """
    rng = random.Random(seed)

    if period_pool_ms is None:
        Ts = [rng.randint(period_min_ms, period_max_ms) for _ in range(n_tasks)]
    else:
        Ts = [rng.choice(period_pool_ms) for _ in range(n_tasks)]

    # Random utilizations that sum to U_total
    w = [rng.random() for _ in range(n_tasks)]
    s = sum(w)
    utils = [U_total * wi / s for wi in w]

    tasks: List[Task] = []
    for i, (T, u) in enumerate(zip(Ts, utils), start=1):
        C = max(1, int(math.ceil(u * T)))
        if C >= T:
            C = max(1, T - 1)

        t_perp = max(1, T - delta_eta_ms)
        t_top = T + delta_eta_ms

        tasks.append(Task(
            name=f"tau_{i}",
            C_ms=int(C),
            T_ms=int(T),
            D_ms=int(T),
            offset_ms=0,
            epsilon_i=float(epsilon),
            J_i=float(J),
            delta_eta_ms=int(delta_eta_ms),
            T_perp_ms=int(t_perp),
            T_top_ms=int(t_top),
        ))
    return tasks


# =========================
# Simulation: trace windows + Anterior/Posterior/Pincer totals
# =========================

def simulate_trace_windows_and_patterns(
    tasks: List[Task],
    *,
    randomized: bool,
    seed: int,
    attacker: str,
    victim: str,
    num_hp_windows: int = 20,
    hp_cap_ms: Optional[int] = 2000,
    signature_mode: str = "order_only",  # "order_only" or "order+dur"
    include_idle_in_trace: bool = True,
) -> TraceResult:
    rng = random.Random(seed)

    hp_true_ms = lcm_list([t.T_ms for t in tasks])
    hp_used_ms = hp_true_ms
    hp_was_capped = False
    if hp_cap_ms is not None and hp_true_ms > hp_cap_ms:
        hp_used_ms = int(hp_cap_ms)
        hp_was_capped = True

    horizon_ms = hp_used_ms * num_hp_windows

    # Per-window raw trace (name, duration_ms)
    win_raw: List[List[Tuple[str, int]]] = [[] for _ in range(num_hp_windows)]

    def add_to_windows(start_ms: int, end_ms: int, task_name: str) -> None:
        if end_ms <= start_ms:
            return
        start_ms = max(0, start_ms)
        end_ms = min(horizon_ms, end_ms)
        cur = start_ms
        while cur < end_ms:
            w = cur // hp_used_ms
            if w >= num_hp_windows:
                break
            w_end = min(end_ms, (w + 1) * hp_used_ms)
            dur = int(w_end - cur)
            if dur > 0:
                win_raw[int(w)].append((task_name, dur))
            cur = w_end

    next_release: Dict[str, int] = {t.name: t.offset_ms for t in tasks}
    job_seq: Dict[str, int] = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    totals = Totals()

    # Strict pattern counting: contiguous segments only; idle breaks
    prev2: Optional[Segment] = None
    prev1: Optional[Segment] = None
    last: Optional[Segment] = None

    def reset_chain() -> None:
        nonlocal prev2, prev1, last
        prev2 = prev1 = last = None

    def release_at(time_ms: int) -> None:
        for t in tasks:
            if next_release[t.name] == time_ms:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                if t.name == attacker:
                    totals.attacker_jobs += 1

                ready.append(Job(
                    task=t,
                    release_ms=time_ms,
                    abs_deadline_ms=time_ms + t.D(),
                    remaining_ms=t.C_ms,
                    job_id=jid
                ))

                if randomized:
                    t_perp = t.T_perp_ms if t.T_perp_ms is not None else max(1, t.T_ms - t.delta_eta_ms)
                    t_top = t.T_top_ms if t.T_top_ms is not None else (t.T_ms + t.delta_eta_ms)
                    ia = bounded_laplace_interarrival_ms(
                        rng=rng,
                        mu_ms=t.T_ms,
                        epsilon=t.epsilon_i,
                        J=t.J_i,
                        delta_eta_ms=t.delta_eta_ms,
                        t_perp_ms=int(t_perp),
                        t_top_ms=int(t_top),
                    )
                    next_release[t.name] = time_ms + ia
                else:
                    next_release[t.name] = time_ms + t.T_ms

    def push_segment(seg: Segment) -> None:
        nonlocal prev2, prev1, last

        # record into trace windows
        add_to_windows(seg.start_ms, seg.end_ms, seg.task_name)

        # merge contiguous same (task, job)
        if last is not None and seg.start_ms == last.end_ms and seg.task_name == last.task_name and seg.job_id == last.job_id:
            last.end_ms = seg.end_ms
            return

        prev2, prev1 = prev1, last
        last = seg

        if seg.task_name == attacker:
            totals.attacker_segments += 1

        # anterior/posterior (strictly contiguous)
        if prev1 is not None and prev1.end_ms == seg.start_ms:
            if prev1.task_name == attacker and seg.task_name == victim:
                totals.anterior += 1
            if prev1.task_name == victim and seg.task_name == attacker:
                totals.posterior += 1

        # strict pincer: A(jobX)->V->A(jobX), contiguous
        if prev2 is not None and prev1 is not None:
            if (
                prev2.task_name == attacker
                and prev1.task_name == victim
                and seg.task_name == attacker
                and prev2.job_id == seg.job_id
                and prev2.end_ms == prev1.start_ms
                and prev1.end_ms == seg.start_ms
            ):
                totals.pincer += 1

    # initial releases at t=0
    t_now = 0
    release_at(0)

    while t_now < horizon_ms:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ms:
                if include_idle_in_trace and t_now < horizon_ms:
                    add_to_windows(t_now, horizon_ms, "IDLE")
                break

            if include_idle_in_trace:
                add_to_windows(t_now, t_next, "IDLE")
            reset_chain()  # idle breaks strict adjacency patterns

            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)

        finish_time = t_now + cur.remaining_ms
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ms)

        ran = t_event - t_now
        if ran > 0:
            push_segment(Segment(
                start_ms=t_now,
                end_ms=t_event,
                task_name=cur.task.name,
                job_id=cur.job_id
            ))
            cur.remaining_ms -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ms == 0:
            ready.remove(cur)

    # compress per-window traces into signatures
    signatures: List[Tuple] = []
    for w in range(num_hp_windows):
        merged: List[List] = []
        for name, dur in win_raw[w]:
            if dur <= 0:
                continue
            if merged and merged[-1][0] == name:
                merged[-1][1] += dur
            else:
                merged.append([name, dur])

        if signature_mode == "order_only":
            signatures.append(tuple(x[0] for x in merged))
        elif signature_mode == "order+dur":
            signatures.append(tuple((x[0], int(x[1])) for x in merged))
        else:
            raise ValueError("signature_mode must be 'order_only' or 'order+dur'")

    freq = Counter(signatures)
    unique = len(freq)

    return TraceResult(
        hp_used_ms=hp_used_ms,
        hp_true_ms=hp_true_ms,
        hp_was_capped=hp_was_capped,
        num_windows=num_hp_windows,
        unique_traces=unique,
        all_windows_match=(unique == 1),
        trace_freq=freq,
        totals=totals,
    )


# =========================
# Main experiment
# =========================

def run_demo(
    *,
    n_tasks: int = 10,
    U_total: float = 0.5,
    num_hp_windows: int = 200,
    hp_cap_ms: Optional[int] = 2000,
    # keep HP manageable; set None for fully random periods (HP may explode)
    period_pool_ms: Optional[List[int]] = [10, 20, 40, 80, 160],
    epsilon: float = 1.0,
    J: float = 16.0,
    delta_eta_ms: int = 100,
    seed: Optional[int] = None,  # None => truly random every run
) -> None:
    if seed is None:
        seed = secrets.randbits(64)

    print(f"seed={seed}")

    # Create taskset RNG seed
    tasks = generate_taskset(
        n_tasks=n_tasks,
        U_total=U_total,
        seed=seed ^ 0xA5A5A5A5A5A5A5A5,
        period_pool_ms=period_pool_ms,
        epsilon=epsilon,
        J=J,
        delta_eta_ms=delta_eta_ms,
    )

    # Pick attacker/victim randomly but deterministically within this run
    names = [t.name for t in tasks]
    pair_rng = random.Random(seed ^ 0x5A5A5A5A5A5A5A5A)
    attacker, victim = pair_rng.sample(names, 2)

    rm_res = simulate_trace_windows_and_patterns(
        tasks,
        randomized=False,
        seed=seed ^ 0x1111111111111111,
        attacker=attacker,
        victim=victim,
        num_hp_windows=num_hp_windows,
        hp_cap_ms=hp_cap_ms,
        signature_mode="order_only",
        include_idle_in_trace=True,
    )

    eps_res = simulate_trace_windows_and_patterns(
        tasks,
        randomized=True,
        seed=seed ^ 0x2222222222222222,
        attacker=attacker,
        victim=victim,
        num_hp_windows=num_hp_windows,
        hp_cap_ms=hp_cap_ms,
        signature_mode="order_only",
        include_idle_in_trace=True,
    )

    def hp_line(res: TraceResult) -> str:
        if res.hp_was_capped:
            return f"HP_true={res.hp_true_ms} ms, window_len_used={res.hp_used_ms} ms (capped)"
        return f"HP_true=HP_used={res.hp_used_ms} ms"

    print(f"tasks={n_tasks}, U={U_total}, windows={num_hp_windows}, attacker={attacker}, victim={victim}")
    print("RM :", hp_line(rm_res))
    print("EPS:", hp_line(eps_res))

    print("\n--- Vanilla RM ---")
    print(f"Unique traces across windows: {rm_res.unique_traces} (all match: {rm_res.all_windows_match})")
    print(f"Anterior={rm_res.totals.anterior}, Posterior={rm_res.totals.posterior}, Pincer={rm_res.totals.pincer}, den(seg)={rm_res.totals.attacker_segments}")

    print("\n--- Epsilon (DPS) ---")
    print(f"Unique traces across windows: {eps_res.unique_traces} (all match: {eps_res.all_windows_match})")
    print(f"Anterior={eps_res.totals.anterior}, Posterior={eps_res.totals.posterior}, Pincer={eps_res.totals.pincer}, den(seg)={eps_res.totals.attacker_segments}")

    print("\nTop-3 most frequent ε traces:")
    for sig, c in eps_res.trace_freq.most_common(3):
        print(f"  count={c}, len={len(sig)}, head={sig[:10]}")


if __name__ == "__main__":
    # Jupyter passes non-numeric args (e.g., --f=...); ignore them.
    seed_arg = None
    for a in sys.argv[1:]:
        try:
            seed_arg = int(a)
            break
        except ValueError:
            continue
    run_demo(seed=seed_arg)

seed=10225620970606762790
tasks=10, U=0.5, windows=200, attacker=tau_8, victim=tau_2
RM : HP_true=HP_used=160 ms
EPS: HP_true=HP_used=160 ms

--- Vanilla RM ---
Unique traces across windows: 1 (all match: True)
Anterior=0, Posterior=0, Pincer=0, den(seg)=1600

--- Epsilon (DPS) ---
Unique traces across windows: 200 (all match: False)
Anterior=19, Posterior=23, Pincer=8, den(seg)=538

Top-3 most frequent ε traces:
  count=1, len=30, head=('tau_10', 'tau_3', 'tau_6', 'tau_7', 'tau_1', 'tau_2', 'tau_5', 'tau_1', 'tau_2', 'tau_1')
  count=1, len=24, head=('IDLE', 'tau_7', 'tau_9', 'IDLE', 'tau_6', 'tau_10', 'tau_3', 'tau_6', 'tau_3', 'tau_6')
  count=1, len=22, head=('IDLE', 'tau_10', 'tau_3', 'tau_6', 'IDLE', 'tau_9', 'tau_5', 'tau_1', 'tau_5', 'tau_9')


# butterfly incidence analysis

In [159]:
import math
import random
import secrets
from dataclasses import dataclass
from typing import List, Optional, Iterable
import pandas as pd


# =========================================================
# Utilities
# =========================================================
def lcm(a: int, b: int) -> int:
    return abs(a // math.gcd(a, b) * b)

def lcm_list(xs: Iterable[int]) -> int:
    hp = 1
    for x in xs:
        hp = lcm(hp, int(x))
    return int(hp)

def laplace_noise(rng: random.Random, b: float) -> float:
    u = rng.random() - 0.5
    if u == 0.0:
        return 0.0
    return -b * math.copysign(1.0, u) * math.log(1.0 - 2.0 * abs(u))

def bounded_laplace_interarrival_ms(
    rng: random.Random,
    mu_ms: int,
    epsilon: float,
    J: float,
    delta_eta_ms: int,
    t_perp_ms: int,
    t_top_ms: int,
) -> int:
    if epsilon <= 0:
        raise ValueError("epsilon must be > 0")

    b = (2.0 * float(J) * float(delta_eta_ms)) / float(epsilon)
    x = float(mu_ms) + laplace_noise(rng, b)

    ia = int(round(x))
    ia = max(1, ia)
    ia = max(t_perp_ms, min(ia, t_top_ms))
    return ia


# =========================================================
# UUniFast
# =========================================================
def uunifast(n_tasks: int, U_total: float, rng: random.Random) -> List[float]:
    """
    Generate n_tasks utilizations that sum to U_total using UUniFast.
    """
    utils: List[float] = []
    sum_u = float(U_total)

    for i in range(1, n_tasks):
        next_sum_u = sum_u * (rng.random() ** (1.0 / (n_tasks - i)))
        utils.append(sum_u - next_sum_u)
        sum_u = next_sum_u

    utils.append(sum_u)
    return utils


# =========================================================
# Data structures
# =========================================================
@dataclass(frozen=True)
class Task:
    name: str
    C_ms: int
    T_ms: int
    D_ms: Optional[int] = None
    offset_ms: int = 0

    epsilon_i: float = 100.0
    J_i: float = 16.0
    delta_eta_ms: int = 100
    T_perp_ms: Optional[int] = None
    T_top_ms: Optional[int] = None

    def D(self) -> int:
        return self.D_ms if self.D_ms is not None else self.T_ms


@dataclass
class Job:
    task: Task
    release_ms: int
    abs_deadline_ms: int
    remaining_ms: int
    job_id: int


@dataclass
class StabilityResult:
    failed: bool
    Rb_ms: float
    Rw_ms: float
    Jitter_ms: float
    lhs: float
    rhs: float


# =========================================================
# RM priority
# =========================================================
def rm_pick(ready: List[Job]) -> Job:
    return min(ready, key=lambda j: (j.task.T_ms, j.release_ms, j.task.name, j.job_id))


# =========================================================
# Taskset generation
# =========================================================
def generate_taskset(
    *,
    n_tasks: int,
    U_total: float,
    seed: int,
    period_pool_ms: Optional[List[int]] = None,
    period_min_ms: int = 50,
    period_max_ms: int = 150,
    epsilon: float = 1.0,
    J: float = 16.0,
    delta_eta_ms: int = 100,
) -> List[Task]:
    rng = random.Random(seed)

    if period_pool_ms is None:
        Ts = [rng.randint(period_min_ms, period_max_ms) for _ in range(n_tasks)]
    else:
        Ts = [rng.choice(period_pool_ms) for _ in range(n_tasks)]

    # Real UUniFast split
    utils = uunifast(n_tasks, U_total, rng)
    rng.shuffle(utils)  # optional: avoid bias from generation order

    tasks: List[Task] = []
    for i, (T, u) in enumerate(zip(Ts, utils), start=1):
        C = max(1, int(math.ceil(u * T)))
        if C >= T:
            C = max(1, T - 1)

        t_perp = max(1, T - delta_eta_ms)
        t_top = T + delta_eta_ms

        tasks.append(
            Task(
                name=f"tau_{i}",
                C_ms=int(C),
                T_ms=int(T),
                D_ms=int(T),
                offset_ms=0,
                epsilon_i=float(epsilon),
                J_i=float(J),
                delta_eta_ms=int(delta_eta_ms),
                T_perp_ms=int(t_perp),
                T_top_ms=int(t_top),
            )
        )
    return tasks


# =========================================================
# Simulate target response times
# =========================================================
def simulate_target_response_times(
    tasks: List[Task],
    *,
    target_task_name: str,
    randomized: bool,
    seed: int,
    num_hp_windows: int = 200,
    hp_cap_ms: Optional[int] = 2000,
) -> List[int]:
    rng = random.Random(seed)

    hp_true_ms = lcm_list([t.T_ms for t in tasks])
    hp_used_ms = hp_true_ms
    if hp_cap_ms is not None and hp_true_ms > hp_cap_ms:
        hp_used_ms = int(hp_cap_ms)

    horizon_ms = hp_used_ms * num_hp_windows

    next_release = {t.name: t.offset_ms for t in tasks}
    job_seq = {t.name: 0 for t in tasks}
    ready: List[Job] = []

    target_response_times: List[int] = []

    def release_at(time_ms: int) -> None:
        for t in tasks:
            if next_release[t.name] == time_ms:
                jid = job_seq[t.name]
                job_seq[t.name] += 1

                ready.append(
                    Job(
                        task=t,
                        release_ms=time_ms,
                        abs_deadline_ms=time_ms + t.D(),
                        remaining_ms=t.C_ms,
                        job_id=jid,
                    )
                )

                if randomized:
                    t_perp = t.T_perp_ms if t.T_perp_ms is not None else max(1, t.T_ms - t.delta_eta_ms)
                    t_top = t.T_top_ms if t.T_top_ms is not None else (t.T_ms + t.delta_eta_ms)

                    ia = bounded_laplace_interarrival_ms(
                        rng=rng,
                        mu_ms=t.T_ms,
                        epsilon=t.epsilon_i,
                        J=t.J_i,
                        delta_eta_ms=t.delta_eta_ms,
                        t_perp_ms=int(t_perp),
                        t_top_ms=int(t_top),
                    )
                    next_release[t.name] = time_ms + ia
                else:
                    next_release[t.name] = time_ms + t.T_ms

    t_now = 0
    release_at(0)

    while t_now < horizon_ms:
        if not ready:
            t_next = min(next_release.values())
            if t_next >= horizon_ms:
                break
            t_now = t_next
            release_at(t_now)
            continue

        cur = rm_pick(ready)
        finish_time = t_now + cur.remaining_ms
        next_arrival = min(next_release.values())
        t_event = min(finish_time, next_arrival, horizon_ms)

        ran = t_event - t_now
        if ran > 0:
            cur.remaining_ms -= ran
            t_now = t_event

        if t_now == next_arrival:
            release_at(t_now)

        if cur.remaining_ms == 0:
            if cur.task.name == target_task_name:
                rt = t_now - cur.release_ms
                target_response_times.append(rt)
            ready.remove(cur)

    return target_response_times


# =========================================================
# Stability rule
# =========================================================
def evaluate_stability_from_response_times(
    response_times: List[int],
    *,
    alpha: float,
    beta: float,
) -> StabilityResult:
    if len(response_times) == 0:
        return StabilityResult(
            failed=True,
            Rb_ms=float("nan"),
            Rw_ms=float("nan"),
            Jitter_ms=float("nan"),
            lhs=float("nan"),
            rhs=beta,
        )

    rb = float(min(response_times))
    rw = float(max(response_times))
    jitter = rw - rb
    lhs = rb + alpha * jitter
    failed = lhs > beta

    return StabilityResult(
        failed=failed,
        Rb_ms=rb,
        Rw_ms=rw,
        Jitter_ms=jitter,
        lhs=lhs,
        rhs=beta,
    )


# =========================================================
# One taskset trial
# =========================================================
def one_taskset_trial(
    *,
    n_tasks: int = 15,
    U_total: float = 0.5,
    period_pool_ms: Optional[List[int]] = (10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150),
    epsilon: float = 1.0,
    J: float = 16.0,
    delta_eta_ms: int = 100,
    alpha: float = 1.0,
    beta: float = 20.83,
    num_hp_windows: int = 200,
    hp_cap_ms: Optional[int] = 2000,
    seed: Optional[int] = None,
):
    if seed is None:
        seed = secrets.randbits(64)

    tasks = generate_taskset(
        n_tasks=n_tasks,
        U_total=U_total,
        seed=seed ^ 0xA5A5A5A5A5A5A5A5,
        period_pool_ms=list(period_pool_ms) if period_pool_ms is not None else None,
        epsilon=epsilon,
        J=J,
        delta_eta_ms=delta_eta_ms,
    )

    names = [t.name for t in tasks]
    pick_rng = random.Random(seed ^ 0x5A5A5A5A5A5A5A5A)
    target_task = pick_rng.choice(names)

    rm_rts = simulate_target_response_times(
        tasks,
        target_task_name=target_task,
        randomized=False,
        seed=seed ^ 0x1111111111111111,
        num_hp_windows=num_hp_windows,
        hp_cap_ms=hp_cap_ms,
    )
    rm_res = evaluate_stability_from_response_times(
        rm_rts,
        alpha=alpha,
        beta=beta,
    )

    eps_rts = simulate_target_response_times(
        tasks,
        target_task_name=target_task,
        randomized=True,
        seed=seed ^ 0x2222222222222222,
        num_hp_windows=num_hp_windows,
        hp_cap_ms=hp_cap_ms,
    )
    eps_res = evaluate_stability_from_response_times(
        eps_rts,
        alpha=alpha,
        beta=beta,
    )

    vanilla_fail = int(rm_res.failed)
    eps_fail = int(eps_res.failed)

    return {
        "seed": int(seed),
        "U": float(U_total),
        "target_task": target_task,

        "Vanilla_failed": vanilla_fail,
        "EPS_failed": eps_fail,

        "Vanilla_only": int(vanilla_fail == 1 and eps_fail == 0),
        "EPS_only": int(vanilla_fail == 0 and eps_fail == 1),
        "Both": int(vanilla_fail == 1 and eps_fail == 1),
        "Neither": int(vanilla_fail == 0 and eps_fail == 0),

        "Vanilla_Rb_ms": rm_res.Rb_ms,
        "Vanilla_Rw_ms": rm_res.Rw_ms,
        "Vanilla_Jitter_ms": rm_res.Jitter_ms,
        "Vanilla_LHS": rm_res.lhs,
        "Vanilla_RHS": rm_res.rhs,

        "EPS_Rb_ms": eps_res.Rb_ms,
        "EPS_Rw_ms": eps_res.Rw_ms,
        "EPS_Jitter_ms": eps_res.Jitter_ms,
        "EPS_LHS": eps_res.lhs,
        "EPS_RHS": eps_res.rhs,
    }


# =========================================================
# Sweep over utilizations
# =========================================================
def run_utilization_sweep(
    *,
    U_values,
    n_runs_per_u: int = 200,
    n_tasks: int = 15,
    period_pool_ms: Optional[List[int]] = (10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150),
    epsilon: float = 1.0,
    J: float = 16.0,
    delta_eta_ms: int = 100,
    alpha: float = 1.0,
    beta: float = 20.83,
    num_hp_windows: int = 200,
    hp_cap_ms: Optional[int] = 2000,
):
    all_rows = []
    summary_rows = []

    global_run_id = 1

    for U_total in U_values:
        rows_u = []

        for local_run_id in range(1, n_runs_per_u + 1):
            res = one_taskset_trial(
                n_tasks=n_tasks,
                U_total=U_total,
                period_pool_ms=period_pool_ms,
                epsilon=epsilon,
                J=J,
                delta_eta_ms=delta_eta_ms,
                alpha=alpha,
                beta=beta,
                num_hp_windows=num_hp_windows,
                hp_cap_ms=hp_cap_ms,
            )
            res["run_global"] = global_run_id
            res["run_within_U"] = local_run_id
            global_run_id += 1

            rows_u.append(res)
            all_rows.append(res)

        df_u = pd.DataFrame(rows_u)

        summary_rows.append({
            "U": float(U_total),
            "n_runs": int(n_runs_per_u),

            "Vanilla_total_violations": int(df_u["Vanilla_failed"].sum()),
            "EPS_total_violations": int(df_u["EPS_failed"].sum()),

            "Vanilla_incidence_percent": 100.0 * df_u["Vanilla_failed"].mean(),
            "EPS_incidence_percent": 100.0 * df_u["EPS_failed"].mean(),

            "Vanilla_only_count": int(df_u["Vanilla_only"].sum()),
            "EPS_only_count": int(df_u["EPS_only"].sum()),
            "Both_count": int(df_u["Both"].sum()),
            "Neither_count": int(df_u["Neither"].sum()),

            "Vanilla_only_percent": 100.0 * df_u["Vanilla_only"].mean(),
            "EPS_only_percent": 100.0 * df_u["EPS_only"].mean(),
            "Both_percent": 100.0 * df_u["Both"].mean(),
            "Neither_percent": 100.0 * df_u["Neither"].mean(),

            "Vanilla_avg_Rb_ms": df_u["Vanilla_Rb_ms"].mean(),
            "Vanilla_avg_Rw_ms": df_u["Vanilla_Rw_ms"].mean(),
            "Vanilla_avg_Jitter_ms": df_u["Vanilla_Jitter_ms"].mean(),

            "EPS_avg_Rb_ms": df_u["EPS_Rb_ms"].mean(),
            "EPS_avg_Rw_ms": df_u["EPS_Rw_ms"].mean(),
            "EPS_avg_Jitter_ms": df_u["EPS_Jitter_ms"].mean(),
        })

    df_all = pd.DataFrame(all_rows)
    df_summary = pd.DataFrame(summary_rows)

    grand_total = pd.DataFrame([{
        "total_tasksets": int(len(df_all)),
        "Vanilla_total_violations": int(df_all["Vanilla_failed"].sum()),
        "EPS_total_violations": int(df_all["EPS_failed"].sum()),

        "Vanilla_incidence_percent": 100.0 * df_all["Vanilla_failed"].mean(),
        "EPS_incidence_percent": 100.0 * df_all["EPS_failed"].mean(),

        "Vanilla_only_count": int(df_all["Vanilla_only"].sum()),
        "EPS_only_count": int(df_all["EPS_only"].sum()),
        "Both_count": int(df_all["Both"].sum()),
        "Neither_count": int(df_all["Neither"].sum()),

        "Vanilla_only_percent": 100.0 * df_all["Vanilla_only"].mean(),
        "EPS_only_percent": 100.0 * df_all["EPS_only"].mean(),
        "Both_percent": 100.0 * df_all["Both"].mean(),
        "Neither_percent": 100.0 * df_all["Neither"].mean(),
    }])

    return df_all, df_summary, grand_total


# =========================================================
# Main
# =========================================================
if __name__ == "__main__":
    U_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

    df_all, df_summary, df_total = run_utilization_sweep(
        U_values=U_values,
        n_runs_per_u=200,
        n_tasks=15,
        period_pool_ms=[10, 20, 40, 80, 160],
        epsilon=1.0,
        J=16.0,
        delta_eta_ms=100,
        alpha=1.0,
        beta=20.83,
        num_hp_windows=200,
        hp_cap_ms=2000,
    )

    print("\n==================== Per-U Summary ====================")
    print(df_summary.to_string(index=False))

    print("\n==================== Grand Total ====================")
    print(df_total.to_string(index=False))

    out_xlsx = "stability_violation_n15.xlsx"
    # out_runs_csv = "stability_violation_all_runs.csv"
    # out_summary_csv = "stability_violation_summary_by_U.csv"
    # out_total_csv = "stability_violation_grand_total.csv"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df_all.to_excel(writer, sheet_name="all_runs", index=False)
        df_summary.to_excel(writer, sheet_name="summary_by_U", index=False)
        df_total.to_excel(writer, sheet_name="grand_total", index=False)

    # df_all.to_csv(out_runs_csv, index=False)
    # df_summary.to_csv(out_summary_csv, index=False)
    # df_total.to_csv(out_total_csv, index=False)

    print("\nSaved files:")
    print(f"  {out_xlsx}")
    # print(f"  {out_runs_csv}")
    # print(f"  {out_summary_csv}")
    # print(f"  {out_total_csv}")


==================== Per-U Summary ====================
  U  n_runs  Vanilla_total_violations  EPS_total_violations  Vanilla_incidence_percent  EPS_incidence_percent  Vanilla_only_count  EPS_only_count  Both_count  Neither_count  Vanilla_only_percent  EPS_only_percent  Both_percent  Neither_percent  Vanilla_avg_Rb_ms  Vanilla_avg_Rw_ms  Vanilla_avg_Jitter_ms  EPS_avg_Rb_ms  EPS_avg_Rw_ms  EPS_avg_Jitter_ms
0.1     200                        11                    58                        5.5                   29.0                   0              47          11            142                   0.0              23.5           5.5             71.0           9.480000           9.480000               0.000000          1.180         15.440             14.260
0.2     200                        32                    84                       16.0                   42.0                   0              52          32            116                   0.0              26.0          16.0         